<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L09-capstone/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/predictive-maintenance/lessons/P03-L09-capstone/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/predictive-maintenance/lessons/P03-L09-capstone/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/predictive-maintenance/lessons/P03-L09-capstone/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P03-L09 · Capstone: a monitoring programme for one asset class

**You will build:** not a model. A **monitoring programme** for one asset class — a fleet
of boiler feed pumps — delivered as this runnable notebook *plus* a one-page specification,
and graded on both. Seven things have to be in it, and every one of them is a decision you
have already made once in an earlier module:

1. the feature, and *why it is causal*;
2. the labelling policy, and *what it censors*;
3. the alarm rule, with its lead time;
4. the three prices, and *where they came from*;
5. the chosen operating point, **with the failures it deliberately gives up named**;
6. the deployment constraint it respects;
7. the re-derivation trigger.

**Time:** ~120 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** every earlier module in this programme. Module 1 gives the sweep, the
lead-time rule and the three prices; module 2 the conditioned signal this lesson starts
from; module 3 the load-and-ambient correction; module 4 the work-order log, suspensions
and right-censoring; modules 5 and 6 a decision priced over a distribution; module 7 the
gateway's arena and its Q8.8 wire format; module 8 the audited prices and the measured
drift alert level.

By the end you will be able to:

1. Implement causal_health and causality_margin so that the claim 'this feature is
   causal' is a number the notebook prints, and measure what a whole-window fit does to
   readings that were already recorded.
2. Implement label_runs and censoring_report so a suspension is censored rather than
   counted as a survivor, and report what the policy censors in units and in unobserved
   unit-hours.
3. Implement price_the_outcomes, choose_operating_point and failures_given_up so the
   threshold is derived in currency from audited prices, with a price the audit could
   not support carried as a labelled guess, and measure, for every failure that
   threshold does not catch, the reason and the price of catching it anyway.
4. Implement deployment_report and rederivation_trigger so the smoothing window is
   decided by the gateway's arena and the re-derivation rule is two measured numbers
   rather than a sentence.
5. Implement check_specification and my_specification, and explain why a submission
   that reports a threshold without its prices, or without naming the failures it
   gives up, must fail however good its detector is.


In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import sys
import time
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# --- the asset class -----------------------------------------------------------------------
ASSET_CLASS = "boiler feed pump, six-stage, 1480 rpm"
N_UNITS = 140
N_HOURS = 720                  # 30 days, hourly
HORIZON_END = N_HOURS - 1      # the last hour the historian holds
SEED = 20260923

# The commissioning window: the ONLY hours any fit in this programme is allowed to see. Every
# hour after it is scored as the plant would score it, with nothing from the future.
BASELINE_HOURS = 120
WINDOW = 11                    # hours of causal median smoothing: the widest the gateway holds
LEAD_HOURS = 48                # module 1's contract: less warning than this is not a catch
TAIL_HOURS = 72                # hours the causality probe in section 3 perturbs

THRESHOLDS = np.round(np.arange(1.00, 3.501, 0.05), 2)

# --- the log, the audit and the prices ------------------------------------------------------
FAILURE = "failure"
SUSPENSION = "suspension"
CENSORED = "censored"
IGNORE = "ignore"
RUN_KINDS = (FAILURE, CENSORED)

OUTCOMES = ("planned", "unplanned", "false_alarm")
AUDITED = "audited"
GUESSED = "guessed"
MIN_OBSERVATIONS = 5           # module 8's rule: fewer jobs than this is not an estimate

# The kick-off guess. Three numbers a room agreed on before anyone had an invoice.
KICKOFF_PLANNED = 15_000.0
KICKOFF_UNPLANNED = 90_000.0
KICKOFF_FALSE_ALARM = 21_000.0

# The grid section 11 walks to find how far a price has to move before the threshold does.
PRICE_MULTIPLIERS = np.round(np.concatenate([np.arange(1.00, 0.49, -0.02),
                                             np.arange(1.02, 2.01, 0.02)]), 2)

# --- drift, carried forward from module 8 ---------------------------------------------------
N_BINS = 12
PSI_FLOOR = 1e-6
NULL_DRAWS = 20

NEVER_ALARMED = "never alarmed"
TOO_LATE = "alarmed too late"

# The two triggers this programme commits to. Your own wording is fine; the checklist grades
# the two NUMBERS beside them, not the prose.
TRIGGERS = ("a price is re-measured", "the feature distribution drifts")


class Prices(NamedTuple):
    """Module 1's cost model, carried through every module since. True negatives are free."""
    planned: float         # a failure caught in time and repaired in a planned window
    unplanned: float       # a failure that happened
    false_alarm: float     # an intervention on a pump that was fine


class Counts(NamedTuple):
    """Module 1's four outcomes. Each field is an int, or an array over many thresholds."""
    tp: Any     # failures caught with at least `lead_hours` of warning
    fp: Any     # alarms on units that never failed inside their run
    fn: Any     # failures not caught in time, whether alarmed late or not at all
    tn: Any     # units that never failed and never alarmed


class OperatingPoint(NamedTuple):
    """Module 1's chosen threshold, and what choosing it implies."""
    index: int
    threshold: float
    cost: float
    accuracy: float
    counts: Counts


class Run(NamedTuple):
    """Module 4's observation window for one unit, after the log became a decision."""
    unit: int
    end_hour: int      # the hour the unit stopped being observed
    kind: str          # FAILURE, or CENSORED for a suspension or an untouched unit


class WorkOrder(NamedTuple):
    """One row of the CMMS export for this asset class."""
    unit: int
    raised_hour: int   # the hour the job was raised — the plant's own stop hour
    closed_hour: int   # the hour the paperwork was closed, often weeks later
    job_type: str      # "BM", "PM", "MOD", "DECOM", "INSP", or "" where nobody filled it in
    text: str          # the fitter's free text


class AuditEntry(NamedTuple):
    """Module 8's alarm audit: one line per job, priced off the invoice."""
    hour: int
    unit: int
    outcome: str       # one of OUTCOMES
    cost: float


class CensoringReport(NamedTuple):
    """What a labelling policy can and cannot see. Exercise 4."""
    n_failures: int
    n_censored: int
    censored_units: tuple
    censored_fraction: float
    unobserved_unit_hours: int


class PriceEvidence(NamedTuple):
    """The three prices, the jobs behind each, and which are still a guess. Exercise 5."""
    prices: Prices
    counts: tuple      # jobs behind each price, in OUTCOMES order
    provenance: tuple  # AUDITED or GUESSED per price, in OUTCOMES order


class GivenUp(NamedTuple):
    """One failure the chosen threshold does not catch, and why. Exercise 7."""
    unit: int
    end_hour: int
    alarm_hour: int      # -1 when the unit never alarmed at this threshold
    warning_hours: int   # end_hour - alarm_hour, or -1 when it never alarmed
    reason: str          # NEVER_ALARMED or TOO_LATE


class GatewayLimits(NamedTuple):
    """Module 7's gateway, as a budget. Nothing is allocated after start-up."""
    arena_bytes: int     # the whole heap the monitoring task gets
    sample_bytes: int    # one retained reading
    constant_bytes: int  # the per-unit constants: baseline level, threshold, alarm latch
    wire_frac: int       # fractional bits on the wire: 8 means Q8.8


class DeploymentReport(NamedTuple):
    """Whether this programme fits the gateway it has to live on. Exercise 8."""
    bytes_per_unit: int
    total_bytes: int
    fits: bool
    wire_threshold: float
    disagreeing_readings: int
    units_whose_alarm_moves: int


class Trigger(NamedTuple):
    """When this threshold stops being valid. Exercise 9."""
    threshold: float
    price_move_fraction: float   # smallest |m - 1| in the grid that moves the threshold
    moved_threshold: float       # what it moves to there
    drift_alert: float           # module 8's measured PSI alert level


class Specification(NamedTuple):
    """The one-page specification. The other half of the deliverable."""
    asset_class: str
    feature: str
    causal_margin: float
    labelling_policy: str
    censors_units: tuple
    lead_hours: int
    threshold: float
    prices: Prices
    price_provenance: tuple
    gives_up: tuple
    deployment_constraint: str
    memory_bytes: int
    rederive_when: tuple
    price_move_fraction: float
    drift_alert: float


class Evidence(NamedTuple):
    """Everything the runnable half measured. The checklist compares the spec with this."""
    causal_margin: float
    censoring: CensoringReport
    lead_hours: int
    point: OperatingPoint
    prices: PriceEvidence
    given_up: tuple
    deployment: DeploymentReport
    trigger: Trigger


class CheckResult(NamedTuple):
    """One line of the submission checklist. Exercise 10."""
    item: str
    ok: bool
    detail: str


CHECKLIST_ITEMS = (
    "feature is causal",
    "labelling policy names what it censors",
    "alarm rule carries its lead time",
    "three prices carry their provenance",
    "threshold is the one the evidence derived",
    "failures given up are named",
    "deployment constraint is respected",
    "re-derivation trigger is stated",
)

KICKOFF = Prices(planned=KICKOFF_PLANNED, unplanned=KICKOFF_UNPLANNED,
                 false_alarm=KICKOFF_FALSE_ALARM)
# 8 KiB of arena for the whole monitoring task, four bytes a retained reading, twelve bytes
# of per-unit constants, and a Q8.8 register map. Module 7's gateway, written down as numbers.
GATEWAY = GatewayLimits(arena_bytes=8192, sample_bytes=4, constant_bytes=12, wire_frac=8)

_FAILED_CHECKS: list = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict = {
    "exercise 1": ("causal_health",),
    "exercise 2": ("causality_margin",),
    "exercise 3": ("label_runs",),
    "exercise 4": ("censoring_report",),
    "exercise 5": ("price_the_outcomes",),
    "exercise 6": ("choose_operating_point",),
    "exercise 7": ("failures_given_up",),
    "exercise 8": ("deployment_report",),
    "exercise 9": ("rederivation_trigger",),
    "exercise 10": ("check_specification",),
    "your specification": ("my_specification",),
}
_STATUS: dict = {}   # label -> "passed" | "failed" | "not started", latest run
_ALL_EXERCISES = tuple(_EXERCISES)
_EVIDENCE_NEEDS = _ALL_EXERCISES[:9]      # everything the evidence bundle is built from


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (label_runs)"; several -> "exercises 3, 6 and 8", and
    ["exercise 2", "your specification"] -> "exercise 2 and your specification"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels if label.startswith("exercise ")]
    items = nums + [label for label in labels if not label.startswith("exercise ")]
    head = "exercises " if len(nums) > 1 else "exercise " if nums else ""
    return head + ", ".join(items[:-1]) + " and " + items[-1]


def _try(label: str, check: Callable, needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    import traceback
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig) -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


def safe_ratio(numerator, denominator, when_zero: float = 0.0):
    """Module 1's elementwise division, returning `when_zero` where the denominator is 0."""
    num = np.asarray(numerator, dtype=float)
    den = np.asarray(denominator, dtype=float)
    out = np.full(np.broadcast(num, den).shape, float(when_zero))
    np.divide(num, den, out=out, where=den != 0)
    return out[()]


print(f"{len(CHECKLIST_ITEMS)} checklist items · {len(_EXERCISES)} exercises · lead "
      f"requirement {LEAD_HOURS} h · commissioning window {BASELINE_HOURS} h")

**The pass condition** is the one this whole programme has argued for: *a threshold
defended in currency, with the failures it accepts named out loud.* The checklist in
section 13 is code you run, not a list you read, and it refuses a specification that
reports a threshold without its prices or without saying what it gives up.

**The data is synthetic and the generator is in this notebook.** Nothing is downloaded and
nothing opens a socket. To grade a programme you have to know which pumps really failed,
when the defect really started and what the invoices really said, and no plant historian
ships with those columns. `meta.yaml` declares the generator under `datasets:`. The method
transfers; these particular numbers describe no real plant.

## 1. What you already own

Eight modules of this programme are behind you, and a capstone that made you rewrite them
would be a test of your typing. So the primitives arrive **given**, exactly as you built
them, and every one carries the module it came from. Read the names; you will assemble
them, and the assembly is where programmes go wrong.

The only thing worth re-reading carefully is `alarm_hours`. It masks each unit's readings
to that unit's own run — a pump that was pulled out on day 12 cannot raise an alarm on day
20, and a monitoring programme that counts one is counting a machine that was in the
workshop.

In [ ]:
# --- from module 1: the sweep, the lead-time rule and the price of a confusion matrix ------
def causal_rolling_median(x: np.ndarray, window: int) -> np.ndarray:
    """Module 1's rolling median along the last axis, looking only backwards. Given to you."""
    if not isinstance(window, (int, np.integer)) or int(window) < 1:
        raise ValueError(f"window must be a positive integer, got {window!r}")
    arr = np.asarray(x, dtype=float)
    win = int(window)
    out = np.empty_like(arr)
    for t in range(arr.shape[-1]):
        out[..., t] = np.median(arr[..., max(0, t - win + 1):t + 1], axis=-1)
    return out


def run_mask(runs: Sequence, n_units: int, n_hours: int) -> np.ndarray:
    """True where a unit was still being observed. Given to you."""
    valid = np.zeros((int(n_units), int(n_hours)), dtype=bool)
    for r in runs:
        valid[int(r.unit), :int(r.end_hour) + 1] = True
    return valid


def alarm_hours(health: np.ndarray, runs: Sequence, threshold: float) -> np.ndarray:
    """Module 1's `alarm_times`, restricted to each unit's own run. Given to you.

    Returns one integer per unit: the first hour whose health index is >= `threshold` while
    the unit was still being observed, or -1 if it never alarmed.
    """
    h = np.asarray(health, dtype=float)
    crossed = (h >= float(threshold)) & run_mask(runs, h.shape[0], h.shape[1])
    return np.where(crossed.any(axis=1), crossed.argmax(axis=1), -1).astype(int)


def classify_runs(alarm_idx: np.ndarray, runs: Sequence,
                  lead_hours: int = LEAD_HOURS) -> Counts:
    """Module 1's `classify_outcomes`, over module 4's run table. Given to you.

    A FAILURE run is a true positive when it alarmed at least `lead_hours` before its end
    hour, inclusive, and a false negative otherwise. A CENSORED run is a false positive if it
    alarmed at all and a true negative otherwise — a suspension is not a negative example,
    but an alarm raised on one still sends a fitter out.
    """
    if int(lead_hours) < 0:
        raise ValueError(f"lead_hours must not be negative, got {lead_hours!r}")
    alarm = np.asarray(alarm_idx)
    tp = fp = fn = tn = 0
    for r in runs:
        a = int(alarm[int(r.unit)])
        if r.kind == FAILURE:
            if a >= 0 and (int(r.end_hour) - a) >= int(lead_hours):
                tp += 1
            else:
                fn += 1
        else:
            fp += 1 if a >= 0 else 0
            tn += 0 if a >= 0 else 1
    return Counts(tp=tp, fp=fp, fn=fn, tn=tn)


def alarm_counts(health: np.ndarray, runs: Sequence, threshold: float,
                 lead_hours: int = LEAD_HOURS) -> Counts:
    """One threshold, four counts. Given to you."""
    return classify_runs(alarm_hours(health, runs, threshold), runs, lead_hours)


def sweep_counts(health: np.ndarray, runs: Sequence, thresholds: np.ndarray,
                 lead_hours: int = LEAD_HOURS) -> Counts:
    """Module 1's threshold sweep: four arrays, one entry per threshold. Given to you."""
    rows = [alarm_counts(health, runs, float(t), lead_hours) for t in np.asarray(thresholds)]
    return Counts(tp=np.array([r.tp for r in rows]), fp=np.array([r.fp for r in rows]),
                  fn=np.array([r.fn for r in rows]), tn=np.array([r.tn for r in rows]))


def expected_cost(counts: Counts, prices: Prices) -> np.ndarray:
    """Module 1's cost: tp * planned + fn * unplanned + fp * false_alarm. Given to you."""
    return (np.asarray(counts.tp, dtype=float) * prices.planned
            + np.asarray(counts.fn, dtype=float) * prices.unplanned
            + np.asarray(counts.fp, dtype=float) * prices.false_alarm)


def accuracy(counts: Counts) -> np.ndarray:
    """Module 1's accuracy, kept so that section 8 can show what it chooses. Given to you."""
    tp = np.asarray(counts.tp, dtype=float)
    fp = np.asarray(counts.fp, dtype=float)
    fn = np.asarray(counts.fn, dtype=float)
    tn = np.asarray(counts.tn, dtype=float)
    return safe_ratio(tp + tn, tp + fp + fn + tn, 0.0)


# --- from module 3: the load and ambient model a residual is taken against -----------------
def fit_duty_model(duty: np.ndarray, ambient: np.ndarray, raw: np.ndarray) -> np.ndarray:
    """Module 3's `fit_load_model`: least squares for `raw ~ a + b*duty + c*ambient`.

    Given to you. Returns the three coefficients as a float array `[a, b, c]`. All three
    inputs are flattened, so a block of hours from many units fits as one population.
    """
    d = np.asarray(duty, dtype=float).ravel()
    a = np.asarray(ambient, dtype=float).ravel()
    y = np.asarray(raw, dtype=float).ravel()
    if not (d.size == a.size == y.size) or d.size == 0:
        raise ValueError(f"duty, ambient and raw must be the same non-empty size; got "
                         f"{d.size}, {a.size}, {y.size}")
    design = np.column_stack([np.ones_like(d), d, a])
    return np.linalg.lstsq(design, y, rcond=None)[0]


def expected_level(duty: np.ndarray, ambient: np.ndarray, coeffs: np.ndarray) -> np.ndarray:
    """Module 3's `expected_temperature`, renamed for vibration. Given to you."""
    c = np.asarray(coeffs, dtype=float)
    return c[0] + c[1] * np.asarray(duty, dtype=float) + c[2] * np.asarray(ambient,
                                                                          dtype=float)


# --- from module 4: the work-order classifier ----------------------------------------------
NO_FAULT_PHRASES = ("no fault found", "nff", "operator error", "false trip")
FAILURE_WORDS = ("seized", "sheared", "broken", "burnt", "failed", "damaged",
                 "tripped on vibration")
SUSPENSION_WORDS = ("removed for", "planned overhaul", "campaign end", "decommission")
FAILURE_JOB_TYPES = ("BM",)
SUSPENSION_JOB_TYPES = ("PM", "MOD", "DECOM")


def classify_work_order(job_type: str, text: str) -> str:
    """Module 4's classifier, unchanged. Given to you.

    A no-fault-found phrase wins over everything: a breakdown call-out that found nothing did
    not end anything. Then a breakdown job type or a failure word means FAILURE, then a
    planned job type or a suspension word means SUSPENSION, and anything else is IGNORE.
    """
    low = str(text).lower()
    job = str(job_type).strip().upper()
    if any(phrase in low for phrase in NO_FAULT_PHRASES):
        return IGNORE
    if job in FAILURE_JOB_TYPES or any(word in low for word in FAILURE_WORDS):
        return FAILURE
    if job in SUSPENSION_JOB_TYPES or any(word in low for word in SUSPENSION_WORDS):
        return SUSPENSION
    return IGNORE


# --- from module 8: the drift statistic and this plant's own alert level -------------------
def reference_bins(values: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Module 8's reference-quantile bin edges. Given to you."""
    v = np.asarray(values, dtype=float).ravel()
    if v.size == 0 or not np.all(np.isfinite(v)) or int(n_bins) < 2:
        raise ValueError("reference_bins needs a finite, non-empty sample and n_bins >= 2")
    cuts = np.quantile(v, np.linspace(0.0, 1.0, int(n_bins) + 1)[1:-1])
    return np.concatenate(([-np.inf], np.unique(cuts), [np.inf]))


def psi(reference: np.ndarray, current: np.ndarray, edges: np.ndarray,
        floor: float = PSI_FLOOR) -> float:
    """Module 8's population stability index over fixed bins. Given to you."""
    ref = np.asarray(reference, dtype=float).ravel()
    cur = np.asarray(current, dtype=float).ravel()
    edge = np.asarray(edges, dtype=float).ravel()
    interior = edge[1:-1]
    n = edge.size - 1
    r = np.bincount(np.searchsorted(interior, ref, side="right"), minlength=n) / ref.size
    c = np.bincount(np.searchsorted(interior, cur, side="right"), minlength=n) / cur.size
    r = np.maximum(r, float(floor))
    c = np.maximum(c, float(floor))
    return float(np.sum((c - r) * np.log(c / r)))


def null_psi_band(reference: np.ndarray, n_bins: int = N_BINS, n_draws: int = NULL_DRAWS,
                  seed: int = SEED) -> float:
    """Module 8's measured alert level: the largest PSI this plant produces when nothing has
    changed. Splits the reference block's UNITS in half at random, not its rows. Given to you.
    """
    rng = np.random.default_rng(seed)
    ref = np.asarray(reference, dtype=float)
    worst = 0.0
    for _ in range(int(n_draws)):
        perm = rng.permutation(ref.shape[0])
        a, b = perm[:ref.shape[0] // 2], perm[ref.shape[0] // 2:]
        left, right = ref[a].ravel(), ref[b].ravel()
        worst = max(worst, psi(left, right, reference_bins(left, n_bins)))
    return float(worst)


def specification_passes(results: Sequence) -> bool:
    """The submission gate. Given to you, and it is one line on purpose.

    Every item on the checklist is required. There is no partial credit on a specification:
    a programme that cannot say what it gives up is not a programme, it is a number.
    """
    return all(bool(r.ok) for r in results)

## 2. The asset class

One hundred and forty boiler feed pumps, one conditioned vibration reading an hour for
thirty days, plus the two process variables the plant already records: **duty** (the load
fraction the pump is running at) and **ambient** (the pump house temperature). Module 2
delivered the reading; module 3 established that a vibration level is a function of duty
and ambient before it is a function of health.

Read the generator. Four things in it decide everything later:

- every pump has its **own size**, so an amplitude threshold across the fleet is really a
  threshold on pump size;
- one pump in eight is a **drifter** that wanders upwards and never fails — the nuisance
  population that the false-alarm price is paid for;
- failing pumps are given a **spread of severities**, on purpose, so that some faults are
  obvious a week out and some are still inside the healthy population's noise the day
  before they break. A fleet where every fault is obvious makes every threshold look good;
- some pumps are **pulled out for reasons that have nothing to do with their condition**.
  Module 4's word for that is a suspension, and it is censored data, not a negative example.

In [ ]:
DUTY_BLOCK = 8            # hours the duty point holds before the operator moves it
DRIFTER_FRACTION = 0.30   # pumps whose level wanders upwards and never fails
WANDER_RANGE = (0.0004, 0.0032)
SHOCK_RATE = 0.020        # unit-hours that START a burst of impulsive shocks
SHOCK_MAX_HOURS = 3       # how long a burst lasts
A0, B0, C0 = 1.00, 0.90, 0.010    # the true duty and ambient law, per unit of pump size
AMB0 = 18.0
N_SLOW_FAILURES = 19      # a defect that grows over days: the ones monitoring is for
N_SUDDEN_FAILURES = 5     # a P-F interval SHORTER than the lead requirement: not catchable
N_FAILURES = N_SLOW_FAILURES + N_SUDDEN_FAILURES
N_SUSPENSIONS = 16
N_NO_FAULT = 26


def generate_fleet(seed: int = SEED, n_units: int = N_UNITS, n_hours: int = N_HOURS):
    """Deterministic synthetic history for one asset class. Nothing is downloaded.

    Returns `(raw, duty, ambient, fail_hour, remove_hour)`:
      * `raw`, `duty`, `ambient` all have shape (n_units, n_hours). `raw` is the conditioned
        vibration RMS in mm/s that module 2 hands over; it is zero after a unit stops.
      * `fail_hour[i]` is the hour unit i failed, or -1.
      * `remove_hour[i]` is the hour unit i was pulled for an unrelated reason, or -1.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(n_hours, dtype=float)

    n_blocks = int(np.ceil(n_hours / DUTY_BLOCK))
    duty = np.repeat(rng.uniform(0.25, 1.00, (n_units, n_blocks)), DUTY_BLOCK,
                     axis=1)[:, :n_hours]
    site = (AMB0 + 6.0 * np.sin(2.0 * np.pi * t / 24.0) + 0.004 * t
            + rng.normal(0.0, 0.4, n_hours))
    ambient = site[None, :] + rng.normal(0.0, 1.5, (n_units, 1))

    size = 2.40 * np.exp(rng.normal(0.0, 0.20, (n_units, 1)))
    level = size * (A0 + B0 * duty + C0 * (ambient - AMB0))

    # The nuisance population: a slow upward wander that never becomes a failure.
    drifter = rng.random(n_units) < DRIFTER_FRACTION
    wander = np.where(drifter, rng.uniform(*WANDER_RANGE, n_units), 0.0)
    level = level * (1.0 + wander[:, None] * t[None, :])

    order = rng.permutation(n_units)
    failing = order[:N_FAILURES]
    suspended = order[N_FAILURES:N_FAILURES + N_SUSPENSIONS]
    healthy = order[N_FAILURES + N_SUSPENSIONS:]

    fail_hour = np.full(n_units, -1, dtype=int)
    remove_hour = np.full(n_units, -1, dtype=int)
    # Severity is the amplitude ratio the defect has reached LEAD_HOURS before the pump
    # breaks — that is, at the last moment an alarm would still have been worth raising. The
    # slow group spans severities that are obvious and severities that are still inside the
    # healthy population's noise at that moment.
    severity = np.linspace(1.10, 3.10, N_SLOW_FAILURES)
    for k, unit in enumerate(failing[:N_SLOW_FAILURES]):
        fh = int(rng.integers(300, n_hours))
        pf = int(rng.integers(180, 340))
        onset = max(BASELINE_HOURS + 24, fh - pf)
        span = fh - onset
        peak = 1.0 + (severity[k] - 1.0) / (((span - LEAD_HOURS) / span) ** 2.0)
        frac = np.clip((t - onset) / span, 0.0, 1.0)
        level[unit] *= 1.0 + (peak - 1.0) * np.where(t >= onset, frac ** 2.0, 0.0)
        fail_hour[unit] = fh
    # The sudden group. The defect is violent — three to six times the healthy amplitude at
    # the end — and its whole P-F interval is shorter than the lead time the maintenance
    # contract asks for. No threshold on this feature catches these in time, and the point of
    # putting them in the fleet is that the specification has to SAY so.
    for unit in failing[N_SLOW_FAILURES:]:
        fh = int(rng.integers(300, n_hours))
        span = int(rng.integers(16, 40))
        onset = fh - span
        peak = float(rng.uniform(3.0, 6.0))
        frac = np.clip((t - onset) / span, 0.0, 1.0)
        level[unit] *= 1.0 + (peak - 1.0) * np.where(t >= onset, frac ** 2.0, 0.0)
        fail_hour[unit] = fh
    for unit in suspended:
        remove_hour[unit] = int(rng.integers(BASELINE_HOURS + 48, n_hours - 1))

    # Impulsive shocks and measurement noise: neither is degradation, both reach the feature.
    # The shocks arrive in short BURSTS of one to three hours — a forklift catches the skid,
    # a relief valve lifts and lifts again — which is why the smoothing window in section 10
    # is a real choice and not a formality. A single outlier any median shrugs off.
    start = rng.random((n_units, n_hours)) < SHOCK_RATE
    length = rng.integers(1, SHOCK_MAX_HOURS + 1, (n_units, n_hours))
    shock = start.copy()
    for k in range(1, SHOCK_MAX_HOURS):
        carried = np.zeros_like(start)
        carried[:, k:] = start[:, :-k] & (length[:, :-k] > k)
        shock |= carried
    level = level * np.where(shock, rng.uniform(2.0, 4.5, (n_units, n_hours)), 1.0)
    raw = level * np.exp(rng.normal(0.0, 0.09, (n_units, n_hours)))

    for unit in range(n_units):
        end = fail_hour[unit] if fail_hour[unit] >= 0 else remove_hour[unit]
        if end >= 0:
            raw[unit, end + 1:] = 0.0     # a pump that has stopped sends nothing
    return raw, duty, ambient, fail_hour, remove_hour


_BM_FAILURE = ("drive end bearing seized, shaft sheared",
               "unit tripped on vibration, coupling damaged",
               "mechanical seal failed, unit off line",
               "thrust bearing burnt, casing opened")
_PLANNED = ("removed for line reconfiguration",
            "planned overhaul, unit out of service",
            "campaign end, pump taken off duty",
            "decommission ahead of tie-in")
_NO_FAULT = ("attended, no fault found, unit restarted",
             "nff - operator error on start permissive",
             "false trip on low suction, reset",
             "investigated vibration alarm, no fault found")


def generate_log(fail_hour: np.ndarray, remove_hour: np.ndarray, seed: int = SEED,
                 n_hours: int = N_HOURS) -> tuple:
    """The CMMS export for this asset class. Deterministic, and deliberately untidy.

    One breakdown order per failure, one planned order per suspension, and a scattering of
    no-fault-found call-outs on pumps that were never sick. Every order carries the hour it
    was RAISED and the hour the paperwork was CLOSED, and the gap between them is the
    bookkeeping decision module 4 priced.
    """
    rng = np.random.default_rng(seed + 1)
    orders = []
    for unit in np.flatnonzero(fail_hour >= 0):
        raised = int(fail_hour[unit])
        closed = min(n_hours - 1, raised + int(rng.integers(8, 430)))
        orders.append(WorkOrder(int(unit), raised, closed, "BM",
                                _BM_FAILURE[int(rng.integers(len(_BM_FAILURE)))]))
    for unit in np.flatnonzero(remove_hour >= 0):
        raised = int(remove_hour[unit])
        closed = min(n_hours - 1, raised + int(rng.integers(4, 260)))
        job = ("PM", "MOD", "DECOM")[int(rng.integers(3))]
        orders.append(WorkOrder(int(unit), raised, closed, job,
                                _PLANNED[int(rng.integers(len(_PLANNED)))]))
    quiet = np.flatnonzero((fail_hour < 0) & (remove_hour < 0))
    for unit in rng.choice(quiet, size=N_NO_FAULT, replace=False):
        raised = int(rng.integers(BASELINE_HOURS, n_hours - 1))
        closed = min(n_hours - 1, raised + int(rng.integers(2, 90)))
        orders.append(WorkOrder(int(unit), raised, closed, "BM",
                                _NO_FAULT[int(rng.integers(len(_NO_FAULT)))]))
    orders.sort(key=lambda o: (o.raised_hour, o.unit))
    return tuple(orders)


# What the invoices actually said. Generator parameters, not a survey of maintenance costs —
# the argument is that the threshold moves when a price is MEASURED, not that these are the
# right levels for anyone's plant.
TRUE_PLANNED = 11_400.0
TRUE_UNPLANNED = 186_000.0
TRUE_FALSE_ALARM = 24_600.0
AUDIT_ROWS = {"planned": 19, "unplanned": 13, "false_alarm": 4}


def generate_audit(seed: int = SEED, n_hours: int = N_HOURS, n_units: int = N_UNITS) -> tuple:
    """Module 8's alarm audit for this asset class: one line per job, priced off the invoice.

    Nineteen planned interventions and thirteen unplanned failures have been written up. The
    false-alarm column has four lines, which is below `MIN_OBSERVATIONS` and is the point:
    the commonest outcome in any alarm log is the one nobody books a cost against.
    """
    rng = np.random.default_rng(seed + 2)
    level = {"planned": TRUE_PLANNED, "unplanned": TRUE_UNPLANNED,
             "false_alarm": TRUE_FALSE_ALARM}
    spread = {"planned": 0.18, "unplanned": 0.22, "false_alarm": 0.30}
    rows = []
    for outcome in OUTCOMES:
        for _ in range(AUDIT_ROWS[outcome]):
            rows.append(AuditEntry(int(rng.integers(0, n_hours)), int(rng.integers(0, n_units)),
                                   outcome,
                                   float(np.round(level[outcome]
                                                  * np.exp(rng.normal(0.0, spread[outcome])),
                                                  0))))
    rows.sort(key=lambda r: (r.hour, r.unit))
    return tuple(rows)


RAW, DUTY, AMBIENT, FAIL_HOUR, REMOVE_HOUR = generate_fleet()
ORDERS = generate_log(FAIL_HOUR, REMOVE_HOUR)
AUDIT = generate_audit()
# The truth the generator knows and the plant does not. Used ONLY to check your labelling
# policy in section 5; nothing that feeds a threshold ever reads it.
TRUTH_RUNS = tuple(Run(u, int(FAIL_HOUR[u]), FAILURE) if FAIL_HOUR[u] >= 0
                   else Run(u, int(REMOVE_HOUR[u]) if REMOVE_HOUR[u] >= 0 else HORIZON_END,
                            CENSORED)
                   for u in range(N_UNITS))

print(f"asset class        {ASSET_CLASS}")
print(f"history            {RAW.shape[0]} pumps x {RAW.shape[1]} hourly readings "
      f"= {RAW.size:,} readings, {RAW.nbytes / 1024**2:.1f} MiB")
print(f"failures           {int((FAIL_HOUR >= 0).sum())} "
      f"({100 * (FAIL_HOUR >= 0).mean():.1f}% of the fleet inside the window)")
print(f"suspensions        {int((REMOVE_HOUR >= 0).sum())} pumps pulled for other reasons")
print(f"work orders        {len(ORDERS)} rows, of which "
      f"{sum(1 for o in ORDERS if classify_work_order(o.job_type, o.text) == IGNORE)} are "
      f"no-fault-found call-outs")
print(f"alarm audit        {len(AUDIT)} priced jobs "
      f"({', '.join(f'{sum(1 for a in AUDIT if a.outcome == o)} {o}' for o in OUTCOMES)})")
print(f"commissioning      hours 0-{BASELINE_HOURS - 1}; the earliest event in the log is "
      f"hour {min(o.raised_hour for o in ORDERS)}")

Two pumps, one that survives and one that does not, in raw mm/s. The point of the picture
is that you cannot see the difference: the sick pump is inside the healthy one's range for
most of the window, and both are dominated by duty.

In [ ]:
def _show_two_pumps() -> None:
    sick = int(np.flatnonzero(FAIL_HOUR >= 0)[np.argmax(FAIL_HOUR[FAIL_HOUR >= 0])])
    well = int(np.flatnonzero((FAIL_HOUR < 0) & (REMOVE_HOUR < 0))[0])
    fig, ax = plt.subplots(figsize=(9.5, 2.8))
    hours = np.arange(N_HOURS)
    ax.plot(hours, RAW[well], lw=0.5, color="0.65", label=f"pump {well}: survives")
    ax.plot(hours, RAW[sick], lw=0.5, color="crimson",
            label=f"pump {sick}: fails at hour {FAIL_HOUR[sick]}")
    ax.axvspan(0, BASELINE_HOURS, color="0.88", zorder=0)
    ax.set_xlabel("hour")
    ax.set_ylabel("conditioned RMS, mm/s")
    ax.set_title("raw vibration; the shaded block is the commissioning window", fontsize=9)
    ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    _show(fig)
    quiet = np.flatnonzero((FAIL_HOUR < 0) & (REMOVE_HOUR < 0))
    call = int(FAIL_HOUR[sick]) - LEAD_HOURS
    pct = float(np.mean(RAW[quiet, call] < RAW[sick, call]))
    print(f"at hour {call} — the last hour an alarm on pump {sick} would still have bought "
          f"{LEAD_HOURS} h of\nwarning — its raw reading sits at the "
          f"{100 * pct:.0f}th percentile of the {quiet.size} healthy pumps.")
    print("a fleet-wide amplitude threshold has to separate those two, and it cannot. That is")
    print("why the first exercise is a feature and not a model.")


_try("two pumps", _show_two_pumps)

## 3. Exercise 1 — `causal_health()`: the feature, and the discipline that makes it causal

Module 3 established the shape: divide out what duty and ambient explain, normalise each
pump against its own quiet level, then smooth causally. The capstone adds one rule, and it
is the rule that decides whether this programme can be deployed at all:

> **Every constant in the feature is fitted on the commissioning window and nothing else.**

The duty-and-ambient coefficients, and each pump's own baseline level, come from hours
`0 .. baseline_hours - 1`. Hour 400 is scored with numbers that existed at hour 119. That
is what "causal" means here, and it is not a stylistic preference: at hour 400 the plant
genuinely does not have hour 401.

<details><summary>💡 Hint 1 — what to think about</summary>

Four steps, in order: what does duty and ambient predict, what is left over, what is this
pump's own quiet level for that leftover, and how do you smooth it without looking
forwards. Which of those four steps is allowed to see an hour past `baseline_hours`? And a
pump three times the size of its neighbour is not three times as sick: does taking out its
own level mean subtracting it or dividing by it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Fit `fit_duty_model` on the first `baseline_hours` columns of all three arrays at once.
Divide the whole of `raw` by `expected_level` over the whole window — that is arithmetic on
constants already fitted, so it is causal. Take each pump's median of that ratio over the
first `baseline_hours` columns, divide by it, and pass the result through
`causal_rolling_median` — in that order, with the smoothing LAST. Validate first: matching
2-D shapes, a `baseline_hours` inside the window, a `window` of at least one hour, and a
strictly positive expected level and baseline level, because both are divisors.
</details>

In [ ]:
def causal_health(raw: np.ndarray, duty: np.ndarray, ambient: np.ndarray,
                  baseline_hours: int = BASELINE_HOURS, window: int = WINDOW) -> np.ndarray:
    """The health index for this asset class: 1.0 on a pump at its commissioning condition.

    Four steps, and every constant in them is fitted on hours `0 .. baseline_hours - 1`:

      1. `coeffs = fit_duty_model(...)` on the commissioning block of all three arrays;
      2. `ratio = raw / expected_level(duty, ambient, coeffs)` over the whole window;
      3. divide each pump's ratio by that pump's own MEDIAN ratio over the commissioning
         block, so a big pump and a small pump both read about 1.0 when they are well;
      4. `causal_rolling_median(..., window)`.

    The ORDER of steps 3 and 4 matters and is part of the specification: the commissioning
    level is the median of the UNSMOOTHED ratio, and the smoothing is the last thing that
    happens. Smoothing first and taking the level from the smoothed block is a different
    feature, and section 11's drift trigger can tell.

    Raise `ValueError` if the three arrays are not the same 2-D shape, if `baseline_hours` is
    not in `1 .. n_hours`, if `window < 1`, or if any expected level or any pump's baseline
    level is not strictly positive — both are divisors, and a silent negative there turns a
    rising fault into a falling one.

    Returns: a (n_units, n_hours) float array, about 1.0 on a healthy pump and rising with a
    defect.

    Example, one pump whose reading is exactly what duty and ambient predict:
        >>> d = np.array([[0.2, 0.8, 0.2, 0.8]])
        >>> a = np.array([[10.0, 10.0, 30.0, 30.0]])
        >>> r = 1.0 + 2.0 * d + 0.1 * a
        >>> causal_health(r, d, a, baseline_hours=4, window=1).round(6).tolist()
        [[1.0, 1.0, 1.0, 1.0]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tempting_health(raw: np.ndarray, duty: np.ndarray, ambient: np.ndarray,
                    baseline_hours: int = BASELINE_HOURS, window: int = WINDOW) -> np.ndarray:
    """The same feature, fitted on everything. Given to you, to measure and to reject.

    Two changes, both of which a backtest rewards: the duty-and-ambient model is fitted on
    the WHOLE window rather than on the commissioning block, and the median filter is centred
    rather than trailing. Neither is available at hour 400 on a live plant, and both are what
    a time-series library gives you by default.
    """
    r = np.asarray(raw, dtype=float)
    d = np.asarray(duty, dtype=float)
    a = np.asarray(ambient, dtype=float)
    coeffs = fit_duty_model(d, a, r)                       # <- the whole window, not the block
    ratio = r / expected_level(d, a, coeffs)
    ratio = ratio / np.median(ratio[:, :int(baseline_hours)], axis=1, keepdims=True)
    half = int(window) // 2
    out = np.empty_like(ratio)
    for t in range(ratio.shape[1]):
        out[:, t] = np.median(ratio[:, max(0, t - half):t + half + 1], axis=1)
    return out


def _check_causal_health() -> None:
    d = np.array([[0.2, 0.8, 0.2, 0.8, 0.5, 0.3, 0.9, 0.6]])
    a = np.array([[10.0, 10.0, 30.0, 30.0, 20.0, 25.0, 15.0, 22.0]])
    r = 1.0 + 2.0 * d + 0.1 * a
    flat = causal_health(r, d, a, baseline_hours=4, window=1)
    assert np.allclose(flat, 1.0), (
        f"a pump whose reading is exactly what duty and ambient predict must read 1.0 at "
        f"every hour; got {np.asarray(flat).round(4).tolist()}. Values that track duty mean "
        "you did not divide by expected_level; values that are not 1.0 at all mean you did "
        "not normalise each pump against its own commissioning median"
    )
    two = np.concatenate([r, 3.0 * r])
    d2, a2 = np.concatenate([d, d]), np.concatenate([a, a])
    both = causal_health(two, d2, a2, baseline_hours=4, window=1)
    assert np.allclose(both, 1.0), (
        f"a pump three times the size of its neighbour is not three times as sick; got "
        f"{np.asarray(both).round(3).tolist()}. The per-pump normalisation is what removes "
        "pump size, and it must divide, not subtract"
    )
    future = two.copy()
    future[:, -1] = 500.0
    head = causal_health(future, d2, a2, baseline_hours=4, window=1)[:, :-1]
    assert np.allclose(head, 1.0), (
        f"changing the LAST hour moved the earlier hours to "
        f"{np.asarray(head).round(3).tolist()} — the feature is reading the future. Fit the "
        "duty model and the per-pump level on the commissioning block only, not on the whole "
        "window"
    )
    ramp = r.copy()
    ramp[:, 4:] *= 2.0
    grew = causal_health(ramp, d, a, baseline_hours=4, window=1)
    assert np.allclose(grew[:, :4], 1.0) and np.allclose(grew[:, 4:], 2.0), (
        f"a pump whose amplitude doubles must read 2.0 after the change and 1.0 before it; "
        f"got {np.asarray(grew).round(3).tolist()}"
    )
    spike = r.copy()
    spike[0, 5] = 900.0
    smoothed = causal_health(spike, d, a, baseline_hours=4, window=3)
    assert smoothed[0, 5] < 5.0, (
        f"one shock at hour 5 pushed the smoothed feature to {smoothed[0, 5]:.1f}; pass the "
        "normalised ratio through causal_rolling_median, which is a median and shrugs a "
        "single spike off"
    )
    # One shock INSIDE the commissioning block, on one of two otherwise identical pumps.
    twin = np.tile(d, 2)
    twin_a = np.tile(a, 2)
    clean = 1.0 + 2.0 * twin + 0.1 * twin_a
    pair = np.concatenate([clean, clean])
    pair[1, 9] *= 3.0
    levels = causal_health(pair, np.concatenate([twin, twin]), np.concatenate([twin_a, twin_a]),
                                baseline_hours=16, window=1)
    quiet = [h for h in range(16) if h != 9]
    assert np.allclose(levels[0, quiet], levels[1, quiet], rtol=0.03), (
        f"two identical pumps, one of which took a single shock during commissioning, now "
        f"read {levels[0, 0]:.4f} and {levels[1, 0]:.4f}. The commissioning level is a "
        "MEDIAN for the same reason the smoothing is: one shock in the baseline window must "
        "not make a pump look healthier for the rest of its life"
    )
    # A step inside the commissioning block, where the two possible orderings of "normalise"
    # and "smooth" give different answers.
    stepped = np.tile(d, 2)
    step_a = np.tile(a, 2)
    level_raw = (1.0 + 2.0 * stepped + 0.1 * step_a) * np.where(np.arange(16) < 8, 1.0, 1.6)
    ordered = causal_health(level_raw, stepped, step_a, baseline_hours=16, window=5)
    assert np.isclose(ordered[0, 0], 1.0 / 1.3) and np.isclose(ordered[0, -1], 1.6 / 1.3), (
        f"this pump's commissioning ratio steps from 1.0 to 1.6, so its commissioning MEDIAN "
        f"is 1.3 and the two halves must read {1.0 / 1.3:.6f} and {1.6 / 1.3:.6f}; got "
        f"{ordered[0, 0]:.6f} and {ordered[0, -1]:.6f}. Reading 1.0 and 1.6 means you "
        "smoothed BEFORE taking the commissioning level: the level comes from the unsmoothed "
        "ratio and the smoothing is the last step"
    )
    for bad in ({"baseline_hours": 0}, {"baseline_hours": 9}, {"window": 0}):
        try:
            causal_health(r, d, a, **{"baseline_hours": 4, "window": 1, **bad})
        except ValueError:
            pass
        else:
            raise AssertionError(f"{bad} must raise ValueError, not be tolerated")
    try:
        causal_health(r, d, a[:, :3], baseline_hours=3, window=1)
    except ValueError:
        pass
    else:
        raise AssertionError("mismatched array shapes must raise ValueError")
    print("exercise 1 looks right — a causal health index, 1.0 at commissioning condition")

In [ ]:
_try("exercise 1", _check_causal_health)

## 4. Exercise 2 — `causality_margin()`: stop asserting it, measure it

"The feature is causal" is the single most common sentence in a monitoring specification
and the least often true. It is also trivially checkable, and the check is the exercise:
**change the future and see whether the past moves.**

Take the history, perturb the last `tail_hours` of every channel as violently as you like,
recompute the feature, and compare the *head* — the hours before the perturbation — with
what it was. A causal feature returns exactly zero. Anything else has told you the number
of mm/s by which your backtest is fiction.

<details><summary>💡 Hint 1 — what to think about</summary>

Which hours must be identical between the two runs, and which are allowed to differ? If you
compared the whole array rather than the head, what would a perfectly causal feature score?
And a feature can read next week's duty point or next week's ambient temperature just as
easily as next week's vibration: which of the three channels does your probe disturb?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Copy the three arrays, overwrite the last `tail_hours` columns of every one of them — raw,
duty AND ambient, not raw alone — with something obviously different, call `feature_fn` on
the originals and on the copies, and return the largest absolute difference over the head
columns only. Validate `tail_hours` first: it is a count of hours at the END, at least one,
and it has to leave at least one hour of head behind.
</details>

In [ ]:
def causality_margin(feature_fn: Callable, raw: np.ndarray, duty: np.ndarray,
                     ambient: np.ndarray, tail_hours: int = TAIL_HOURS) -> float:
    """The largest amount by which perturbing the future moves the past. Zero, or a defect.

    Replace the last `tail_hours` columns of `raw` with four times their value plus three,
    of `duty` with 1.0 and of `ambient` with their value plus 20.0; call `feature_fn(raw,
    duty, ambient)` on the untouched arrays and on the perturbed ones; return
    `max |perturbed - original|` over the first `n_hours - tail_hours` columns.

    Raise `ValueError` if `tail_hours` is not in `1 .. n_hours - 1`, because a probe that
    perturbs everything leaves no head to read.

    Returns: a float. Exactly 0.0 for a causal feature.

    Example, a feature that only ever reads the current hour:
        >>> causality_margin(lambda r, d, a: r * 1.0, np.ones((1, 4)), np.ones((1, 4)),
        ...                  np.ones((1, 4)), tail_hours=2)
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_causality_margin() -> None:
    ones = np.ones((1, 4))
    assert causality_margin(lambda r, d, a: r * 1.0, ones, ones, ones, tail_hours=2) == 0.0, (
        "a feature that reads only the current hour has a margin of exactly 0.0"
    )
    leak = causality_margin(lambda r, d, a: np.broadcast_to(r.mean(axis=1, keepdims=True),
                                                            r.shape).copy(),
                            ones, ones, ones, tail_hours=2)
    assert leak > 0.0, (
        f"a feature that averages the WHOLE row into every hour must score above zero; got "
        f"{leak}. Comparing the whole array instead of the head hides nothing and reveals "
        "nothing — the tail is supposed to change"
    )
    whole = causality_margin(lambda r, d, a: r * 1.0, ones * 2.0, ones, ones, tail_hours=1)
    assert whole == 0.0, (
        f"the causal identity feature scored {whole} on a one-hour tail; you are comparing "
        "the perturbed columns, which are meant to differ, instead of the head"
    )
    uses_duty = causality_margin(lambda r, d, a: np.broadcast_to(d.max(axis=1, keepdims=True),
                                                                 d.shape).copy(),
                                 ones, ones * 0.1, ones, tail_hours=2)
    assert uses_duty > 0.0, (
        f"a feature that reads the whole DUTY row leaked and scored {uses_duty}; the probe "
        "has to perturb all three channels, not just raw"
    )
    uses_ambient = causality_margin(
        lambda r, d, a: np.broadcast_to(a.max(axis=1, keepdims=True), a.shape).copy(),
        ones, ones, ones, tail_hours=2)
    assert uses_ambient > 0.0, (
        f"a feature that reads the whole AMBIENT row leaked and scored {uses_ambient}; "
        "perturb ambient too"
    )
    def reads_one_hour_ahead(r, d, a):
        return np.concatenate([r[:, 1:], r[:, -1:]], axis=1)

    long_run = np.ones((1, 20))
    single = causality_margin(reads_one_hour_ahead, long_run, long_run, long_run,
                              tail_hours=5)
    assert np.isclose(single, 6.0), (
        f"a feature that reads one hour ahead moves exactly one head reading, from 1.0 to "
        f"7.0, so the margin is 6.0; got {single}. A value near 0.4 means you averaged the "
        "change over the head instead of taking the WORST of it"
    )
    for bad in (0, -1, 4, 9):
        try:
            causality_margin(lambda r, d, a: r, ones, ones, ones, tail_hours=bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"tail_hours={bad} on a 4-hour history must raise ValueError")
    print("exercise 2 looks right — the probe perturbs the future and reads the past")

In [ ]:
_try("exercise 2", _check_causality_margin)

Now run the probe on your own feature and on the one that fits everything to everything.
The second is not a strawman: a centred filter and a whole-window normaliser are what you
get by default from almost every time-series library, and they are what a backtest written
in an afternoon contains.

In [ ]:
HEALTH = None
TEMPTING = None


def _measure_the_leak() -> None:
    global HEALTH, TEMPTING
    HEALTH = causal_health(RAW, DUTY, AMBIENT)
    TEMPTING = tempting_health(RAW, DUTY, AMBIENT)
    mine = causality_margin(causal_health, RAW, DUTY, AMBIENT)
    theirs = causality_margin(tempting_health, RAW, DUTY, AMBIENT)
    print(f"  {'feature':<34}{'causality margin':>18}{'verdict':>16}")
    print(f"  {'causal_health (yours)':<34}{mine:>18.10f}"
          f"{'deployable' if mine == 0.0 else 'LEAKS':>16}")
    print(f"  {'tempting_health (fitted on all)':<34}{theirs:>18.10f}"
          f"{'deployable' if theirs == 0.0 else 'LEAKS':>16}")
    print(f"\nthe probe changed hours {N_HOURS - TAIL_HOURS}-{N_HOURS - 1} and read hours "
          f"0-{N_HOURS - TAIL_HOURS - 1}.")
    span = float(THRESHOLDS[-1]) - 1.0
    print(f"your feature did not move. The other one moved by up to {theirs:.4f} health "
          f"units at hours\nthat had already been recorded — "
          f"{theirs / span:.1f} times the whole span of the threshold grid "
          f"(1.00 to {THRESHOLDS[-1]:.2f}).")
    head = slice(0, N_HOURS - TAIL_HOURS)
    raw2, duty2, amb2 = RAW.copy(), DUTY.copy(), AMBIENT.copy()
    raw2[:, head.stop:] = 4.0 * raw2[:, head.stop:] + 3.0
    duty2[:, head.stop:] = 1.0
    amb2[:, head.stop:] = amb2[:, head.stop:] + 20.0
    shifted = np.abs(tempting_health(raw2, duty2, amb2)[:, head] - TEMPTING[:, head])
    print(f"and it is not one freak reading: after the probe, "
          f"{100 * float(np.mean(shifted > 0.01)):.0f}% of the already-recorded\nreadings "
          f"come back different by more than 0.01, and "
          f"{100 * float(np.mean(shifted > 0.10)):.0f}% by more than 0.10.")


_try("the causality probe", _measure_the_leak, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — `label_runs()`: the labelling policy

Module 4's argument, in one function. The log is the only record of what happened, and
three decisions inside it move every number downstream:

- **which hour is the event** — the hour the job was raised, or the hour the paperwork was
  closed. This plant closes jobs weeks late, so the two policies disagree by weeks;
- **what a planned removal is** — a suspension, which is censored data. A pump pulled for a
  line reconfiguration is not a pump that survived; it is a pump nobody watched after that
  hour, and calling it a healthy negative teaches the detector that a pump about to be
  pulled looks fine;
- **what a no-fault-found call-out is** — nothing. `classify_work_order` already drops
  them; your job is to honour that and not to let an IGNORE order end a run.

<details><summary>💡 Hint 1 — what to think about</summary>

Every unit gets exactly one run, including the ones the log never mentions — what is their
end hour, and what kind are they? Each order carries two hours: which one belongs to the
pump, and which to the maintenance office? If a unit has two orders, which one ends the
run? And if two orders on the same unit share an hour, which kind wins?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate first: `n_units >= 1`, `horizon_end >= 0`, and every order's unit and raised hour
inside those bounds — `horizon_end` itself is a legal hour, one past it is not. Start every
unit as a run censored at `horizon_end`. Classify each order: an IGNORE ends nothing, and a
SUSPENSION ends the run as CENSORED. Sort what is left so that earlier raised hours come
first and, on an equal hour, FAILURE comes before SUSPENSION; then let the first surviving
order per unit set that unit's run. Return them sorted by unit.
</details>

In [ ]:
def label_runs(orders: Sequence, n_units: int = N_UNITS,
               horizon_end: int = HORIZON_END) -> tuple:
    """Turn the CMMS export into one `Run` per unit, under the raised-hour policy.

    * an order classified FAILURE ends the run as `Run(unit, raised_hour, FAILURE)`;
    * an order classified SUSPENSION ends it as `Run(unit, raised_hour, CENSORED)` — a
      suspension is censored data, not a negative example;
    * an order classified IGNORE ends nothing;
    * a unit the log never mentions is `Run(unit, horizon_end, CENSORED)`;
    * where a unit has several orders the EARLIEST raised hour wins, and on a tie FAILURE
      beats SUSPENSION, because a defect found during planned work is still a defect.

    Raise `ValueError` if `n_units < 1`, if `horizon_end < 0`, or if any order names a unit
    outside `0 .. n_units - 1` or a raised hour outside `0 .. horizon_end`.

    Returns: a tuple of exactly `n_units` `Run`s, sorted by unit.

    Example:
        >>> log = (WorkOrder(1, 5, 40, "BM", "mechanical seal failed, unit off line"),
        ...        WorkOrder(2, 7, 9, "BM", "attended, no fault found, unit restarted"))
        >>> label_runs(log, n_units=3, horizon_end=99)
        (Run(unit=0, end_hour=99, kind='censored'), Run(unit=1, end_hour=5, kind='failure'), Run(unit=2, end_hour=99, kind='censored'))
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_label_runs() -> None:
    log = (WorkOrder(1, 5, 40, "BM", "mechanical seal failed, unit off line"),
           WorkOrder(2, 7, 9, "BM", "attended, no fault found, unit restarted"))
    got = label_runs(log, n_units=3, horizon_end=99)
    assert got == (Run(0, 99, CENSORED), Run(1, 5, FAILURE), Run(2, 99, CENSORED)), (
        f"expected unit 0 censored at the horizon, unit 1 a failure at hour 5 and unit 2 "
        f"censored at the horizon; got {got}. Unit 2's call-out found no fault, so it ended "
        "nothing; unit 0 is never mentioned and is still a run"
    )
    assert len(label_runs((), n_units=4, horizon_end=10)) == 4, (
        "an empty log still produces one run per unit, every one censored at the horizon — "
        "a unit nobody wrote about is not a unit that does not exist"
    )
    susp = label_runs((WorkOrder(0, 12, 50, "PM", "planned overhaul, unit out of service"),),
                      n_units=1, horizon_end=99)
    assert susp == (Run(0, 12, CENSORED),), (
        f"a planned removal is a SUSPENSION, and a suspension is right-censored at the hour "
        f"it was raised: expected Run(0, 12, '{CENSORED}'), got {susp}"
    )
    first = label_runs((WorkOrder(0, 60, 61, "BM", "thrust bearing burnt, casing opened"),
                        WorkOrder(0, 20, 90, "MOD", "removed for line reconfiguration")),
                       n_units=1, horizon_end=99)
    assert first == (Run(0, 20, CENSORED),), (
        f"the EARLIEST surviving order ends the run; expected the hour-20 removal, got "
        f"{first}. Taking the last order backdates nothing and post-dates everything"
    )
    tie = label_runs((WorkOrder(0, 30, 90, "PM", "campaign end, pump taken off duty"),
                      WorkOrder(0, 30, 33, "BM", "drive end bearing seized, shaft sheared")),
                     n_units=1, horizon_end=99)
    assert tie == (Run(0, 30, FAILURE),), (
        f"on a tie in raised hour a FAILURE beats a SUSPENSION — a defect found during "
        f"planned work is still a defect; got {tie}"
    )
    late = label_runs((WorkOrder(0, 5, 88, "BM", "mechanical seal failed, unit off line"),),
                      n_units=1, horizon_end=99)
    assert late == (Run(0, 5, FAILURE),), (
        f"this policy is the RAISED hour, not the closed hour; expected end_hour 5, got "
        f"{late}. The closed hour is when somebody typed, not when the pump stopped"
    )
    for bad in ((log, 0, 99), (log, 3, -1), ((WorkOrder(9, 1, 2, "BM", "seized"),), 3, 99),
                ((WorkOrder(0, 400, 401, "BM", "seized"),), 3, 99)):
        try:
            label_runs(bad[0], n_units=bad[1], horizon_end=bad[2])
        except ValueError:
            pass
        else:
            raise AssertionError(f"label_runs{bad[1:]} on that log must raise ValueError")
    print("exercise 3 looks right — one run per unit, suspensions censored, call-outs ignored")

In [ ]:
_try("exercise 3", _check_label_runs)

## 6. Exercise 4 — `censoring_report()`: what the policy cannot see

The specification has to say what the labelling policy **censors**, and "some units are
censored" is not a statement anybody can act on. Two numbers make it one: which pumps, and
how many unit-hours of operation nobody observed because of it. The second is the honest
measure of how much of this fleet's history the programme is blind to.

<details><summary>💡 Hint 1 — what to think about</summary>

A unit censored at the very last hour of the window hid nothing from you. A unit censored
on day 12 of 30 hid eighteen days. What is the difference between those two, in hours, and
what is it for a fleet?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate the run table first: non-empty, `n_hours >= 1`, every kind known, every end hour
inside the window — whose last hour is one less than `n_hours`. Then count the two kinds,
collect the censored units in sorted order, and divide the censored count by the total.
For the unobserved hours, walk the CENSORED runs only and add up the hours between each
one's end and the last hour of the window: a failure adds nothing, and a run censored at
the last hour adds zero.
</details>

In [ ]:
def censoring_report(runs: Sequence, n_hours: int = N_HOURS) -> CensoringReport:
    """What a run table observed and what it did not.

    * `n_failures`, `n_censored`: how many runs of each kind;
    * `censored_units`: their unit ids as a tuple, in increasing order;
    * `censored_fraction`: `n_censored / len(runs)`;
    * `unobserved_unit_hours`: summed over CENSORED runs only,
      `n_hours - 1 - end_hour` — the hours of operation the programme can say nothing about.
      A run censored at the last hour contributes zero; it hid nothing.

    Raise `ValueError` if `runs` is empty, if `n_hours < 1`, if any kind is not in
    `RUN_KINDS`, or if any end hour is outside `0 .. n_hours - 1`.

    Returns: a `CensoringReport`.

    Example, three units over a 100-hour window:
        >>> censoring_report((Run(0, 99, CENSORED), Run(1, 40, FAILURE), Run(2, 60, CENSORED)),
        ...                  n_hours=100)
        CensoringReport(n_failures=1, n_censored=2, censored_units=(0, 2), censored_fraction=0.6666666666666666, unobserved_unit_hours=39)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_censoring_report() -> None:
    got = censoring_report((Run(0, 99, CENSORED), Run(1, 40, FAILURE), Run(2, 60, CENSORED)),
                           n_hours=100)
    assert got.n_failures == 1 and got.n_censored == 2, (
        f"one failure and two censored runs; got {got.n_failures} and {got.n_censored}"
    )
    assert got.censored_units == (0, 2), (
        f"censored_units must be a sorted tuple of unit ids, got {got.censored_units!r}"
    )
    assert np.isclose(got.censored_fraction, 2 / 3), (
        f"censored_fraction is 2/3 of the RUNS, got {got.censored_fraction}"
    )
    assert got.unobserved_unit_hours == 39, (
        f"expected 39 unobserved unit-hours — unit 0 ends at the last hour and hid nothing, "
        f"unit 2 ends at hour 60 of 0..99 and hid 39 — got {got.unobserved_unit_hours}. "
        "Counting the FAILURE run's remaining hours too would give 98: a pump that broke did "
        "not hide its failure from you"
    )
    none = censoring_report((Run(0, 10, FAILURE),), n_hours=11)
    assert none.censored_units == () and none.unobserved_unit_hours == 0, (
        f"a fleet with no censored runs reports an empty tuple and zero hours, got {none}"
    )
    for bad in (((), 10), ((Run(0, 5, FAILURE),), 0), ((Run(0, 5, "suspension"),), 10),
                ((Run(0, 50, FAILURE),), 10), ((Run(0, -1, FAILURE),), 10)):
        try:
            censoring_report(bad[0], n_hours=bad[1])
        except ValueError:
            pass
        else:
            raise AssertionError(f"censoring_report({bad[0]!r}, {bad[1]}) must raise ValueError")
    print("exercise 4 looks right — the censored units, and the hours nobody watched")

In [ ]:
_try("exercise 4", _check_censoring_report)

Run this once exercises 3 and 4 pass. It scores your policy against the truth the generator
knows — which is a luxury this notebook has and your plant does not — and against the
policy most CMMS exports invite, which is to use the hour the paperwork was closed.

In [ ]:
RUNS = None


def closed_hour_runs() -> tuple:
    """Module 4's 'closed' policy, for comparison only. Given to you."""
    ends = {u: Run(u, HORIZON_END, CENSORED) for u in range(N_UNITS)}
    for o in sorted(ORDERS, key=lambda o: o.closed_hour):
        kind = classify_work_order(o.job_type, o.text)
        if kind == IGNORE or ends[o.unit].end_hour != HORIZON_END:
            continue
        ends[o.unit] = Run(o.unit, int(o.closed_hour),
                           FAILURE if kind == FAILURE else CENSORED)
    return tuple(ends[u] for u in range(N_UNITS))


def _show_the_policy() -> None:
    global RUNS
    RUNS = label_runs(ORDERS)
    mine = censoring_report(RUNS)
    truth = censoring_report(TRUTH_RUNS)
    closed = censoring_report(closed_hour_runs())
    print(f"  {'policy':<22}{'failures':>10}{'censored':>10}{'unobserved unit-h':>20}"
          f"{'median error, h':>18}")
    for name, runs, rep in (("raised hour (yours)", RUNS, mine),
                            ("closed hour", closed_hour_runs(), closed),
                            ("the truth", TRUTH_RUNS, truth)):
        err = np.median([abs(int(a.end_hour) - int(b.end_hour))
                         for a, b in zip(runs, TRUTH_RUNS) if b.kind == FAILURE])
        print(f"  {name:<22}{rep.n_failures:>10}{rep.n_censored:>10}"
              f"{rep.unobserved_unit_hours:>20,}{err:>18.0f}")
    agree = sum(1 for a, b in zip(RUNS, TRUTH_RUNS) if a == b)
    print(f"\nyour policy reproduces {agree} of {N_UNITS} runs exactly. The closed-hour "
          f"policy books every\nfailure late by the median above, which is the paperwork "
          f"lag and not a property of any pump.")
    early = sum(1 for r in RUNS if r.kind == CENSORED and r.end_hour < HORIZON_END)
    print(f"{mine.n_censored} of {N_UNITS} runs are censored: {early} pumps pulled before "
          f"the window closed and\n{mine.n_censored - early} still running at the end of "
          f"it. Only the first group hides anything, and it hides\n"
          f"{mine.unobserved_unit_hours:,} unit-hours — "
          f"{100 * mine.unobserved_unit_hours / (N_UNITS * N_HOURS):.1f}% of this fleet's "
          f"record. That sentence belongs in the specification;\n'some units are censored' "
          f"does not.")


_try("the labelling policy", _show_the_policy, needs=("exercise 3", "exercise 4"))

## 7. Exercise 5 — `price_the_outcomes()`: the three prices, and where they came from

Module 8 built this and the capstone needs one thing added to it: **provenance**. A price
that came off thirteen invoices and a price that came off a whiteboard are both floats, and
the specification has to be able to tell them apart, because the threshold inherits the
status of the weakest one.

The rule is module 8's: a class with fewer than `min_observations` jobs keeps the kick-off
guess **and is reported as a guess**. It does not get the mean of two invoices, and it
certainly does not get zero — pricing a missed failure at nothing makes "never alarm" the
cost-optimal policy, which is an answer this programme will happily produce if you let it.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the mean of an empty list, and what does a monitoring programme do with it? Which
of the three outcomes is the audit most likely to be short of, and what does under-pricing
that one do to the threshold? And breakdown invoices have a long right tail: which of the
two usual averages does that tail move, and what does the other do to that price?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate every row first — known outcome, finite non-negative cost — then
`min_observations >= 1` and a finite, non-negative fallback. Collect the costs per outcome
in `OUTCOMES` order. For each, if there are at least `min_observations` of them take their
arithmetic mean and mark it audited; otherwise take the matching field of `fallback` and
mark it guessed. Build the `Prices` from the three in order.
</details>

In [ ]:
def price_the_outcomes(audit: Sequence, fallback: Prices = KICKOFF,
                       min_observations: int = MIN_OBSERVATIONS) -> PriceEvidence:
    """Module 8's audited prices, with the provenance the specification has to carry.

    For each outcome in `OUTCOMES` order: if the audit holds at least `min_observations`
    lines for it, the price is the arithmetic MEAN of their costs and the provenance is
    `AUDITED`; otherwise the price is that field of `fallback` and the provenance is
    `GUESSED`. `counts` reports what the audit HOLDS, not what was used, so a reader can see
    how close a guessed price came to being an estimate.

    Raise `ValueError` if any row's outcome is not in `OUTCOMES`, if any cost is negative or
    not finite, if `min_observations < 1`, or if any fallback price is negative or not finite.

    Returns: a `PriceEvidence`.

    Example, an audit that can price one outcome and not the others:
        >>> rows = (AuditEntry(1, 0, "planned", 10.0), AuditEntry(2, 1, "planned", 20.0))
        >>> ev = price_the_outcomes(rows, Prices(1.0, 2.0, 3.0), min_observations=2)
        >>> ev.prices, ev.counts, ev.provenance
        (Prices(planned=15.0, unplanned=2.0, false_alarm=3.0), (2, 0, 0), ('audited', 'guessed', 'guessed'))
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_price_the_outcomes() -> None:
    rows = (AuditEntry(1, 0, "planned", 10.0), AuditEntry(2, 1, "planned", 20.0))
    ev = price_the_outcomes(rows, Prices(1.0, 2.0, 3.0), min_observations=2)
    assert ev.prices == Prices(15.0, 2.0, 3.0), (
        f"expected the mean of the two planned invoices and the fallback for the other two; "
        f"got {ev.prices}"
    )
    assert ev.counts == (2, 0, 0), (
        f"counts report what the AUDIT holds, in OUTCOMES order; got {ev.counts!r}"
    )
    assert ev.provenance == (AUDITED, GUESSED, GUESSED), (
        f"provenance is one label per price, in OUTCOMES order; got {ev.provenance!r}"
    )
    short = price_the_outcomes(rows, Prices(1.0, 2.0, 3.0), min_observations=3)
    assert short.prices.planned == 1.0 and short.provenance[0] == GUESSED, (
        f"two invoices against a minimum of three is not an estimate: the price stays at the "
        f"fallback 1.0 and is reported as {GUESSED!r}; got {short.prices.planned} and "
        f"{short.provenance[0]!r}. A mean of an empty or thin class is how a missed failure "
        "ends up priced at nothing and 'never alarm' becomes optimal"
    )
    skew = price_the_outcomes(tuple(AuditEntry(i, i, "unplanned", c) for i, c in
                                    enumerate([10.0, 10.0, 10.0, 10.0, 60.0])),
                              Prices(1.0, 2.0, 3.0), min_observations=5)
    assert np.isclose(skew.prices.unplanned, 20.0), (
        f"the price is the MEAN of the invoices, which is 20.0 on this right-skewed class; "
        f"got {skew.prices.unplanned}. A median would give 10.0 and would systematically "
        "under-price the outcome whose tail is the reason anyone is monitoring"
    )
    for bad_audit, bad_kwargs in (
            ((AuditEntry(1, 0, "planned_repair", 10.0),), {}),
            ((AuditEntry(1, 0, "planned", -5.0),), {}),
            ((AuditEntry(1, 0, "planned", np.nan),), {}),
            (rows, {"min_observations": 0}),
            (rows, {"fallback": Prices(1.0, -2.0, 3.0)}),
            (rows, {"fallback": Prices(1.0, np.inf, 3.0)})):
        try:
            price_the_outcomes(bad_audit, **{"fallback": Prices(1.0, 2.0, 3.0),
                                             "min_observations": 1, **bad_kwargs})
        except ValueError:
            pass
        else:
            raise AssertionError(f"price_the_outcomes({bad_audit!r}, {bad_kwargs!r}) must "
                                 "raise ValueError")
    print("exercise 5 looks right — audited where the evidence exists, a labelled guess where "
          "it does not")

In [ ]:
_try("exercise 5", _check_price_the_outcomes)

Here is what this plant's audit can and cannot price. Note which column is short: it is
always the same one, on every plant, because a call-out that found nothing is the job
nobody bothers to write up.

In [ ]:
def _show_the_prices() -> None:
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"  {'outcome':<14}{'kick-off guess':>16}{'audited price':>16}{'jobs':>7}"
          f"{'provenance':>12}")
    for name, guess, price, n, prov in zip(OUTCOMES, KICKOFF, ev.prices, ev.counts,
                                           ev.provenance):
        print(f"  {name:<14}{guess:>16,.0f}{price:>16,.0f}{n:>7}{prov:>12}")
    worst = max(zip(OUTCOMES, KICKOFF, ev.prices), key=lambda r: abs(r[2] / r[1] - 1.0))
    print(f"\nthe kick-off guess for a {worst[0]} outcome was out by a factor of "
          f"{worst[2] / worst[1]:.2f}.")
    still = [n for n, p in zip(OUTCOMES, ev.provenance) if p == GUESSED]
    print(f"still a guess: {', '.join(still) if still else 'none of them'} — "
          f"{ev.counts[OUTCOMES.index(still[0])] if still else 0} jobs against a minimum of "
          f"{MIN_OBSERVATIONS}.")
    print("that line goes on the specification. A threshold is only as defensible as its "
          "weakest price.")


_try("the three prices", _show_the_prices, needs=("exercise 5",))

## 8. Exercise 6 — `choose_operating_point()`: the threshold, in currency

Module 1's chooser, with the objective removed. This programme does not offer an accuracy
option, because a monitoring programme that chooses by accuracy is choosing a threshold
nobody can defend to the person who signs the work order. The sweep is given; the choice is
yours, and so is the tie rule.

<details><summary>💡 Hint 1 — what to think about</summary>

Cost is minimised, not maximised. On a tie between two thresholds, which one do you want —
the one that alarms more readily or the one that alarms less? What has to be checked before
you index anything, so that a mismatch cannot report the right counts against the wrong
threshold?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check that every field of `counts` is as long as `thresholds` and that the grid is
non-empty, then price the whole sweep with `expected_cost`, take `argmin` — which already
breaks ties towards the lowest index, and therefore the lowest, more cautious threshold —
and build the `OperatingPoint` at that index out of plain ints and floats. The accuracy it
reports is the accuracy AT that index, not the best accuracy anywhere on the grid.
</details>

In [ ]:
def choose_operating_point(thresholds: np.ndarray, counts: Counts,
                           prices: Prices) -> OperatingPoint:
    """Pick the cost-optimal threshold from a swept confusion matrix.

    There is no objective argument: this programme chooses in currency. On a tie, take the
    LOWEST threshold — of two equally priced choices, the one that alarms earlier is the one
    you can defend. `numpy.argmin` already does that.

    `counts` in the returned point is a `Counts` of plain ints at the chosen threshold alone,
    and `accuracy` is reported so the specification can say what it is rather than pretend it
    was not looked at.

    Raise `ValueError` if `thresholds` is empty, or if any field of `counts` is not the same
    length as `thresholds`.

    Returns: an `OperatingPoint`.

    Example:
        >>> c = Counts(tp=np.array([2, 1]), fp=np.array([8, 0]),
        ...            fn=np.array([0, 1]), tn=np.array([90, 98]))
        >>> p = Prices(planned=10.0, unplanned=1000.0, false_alarm=1.0)
        >>> choose_operating_point(np.array([1.0, 2.0]), c, p).threshold
        1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_choose_operating_point() -> None:
    c = Counts(tp=np.array([2, 1]), fp=np.array([8, 0]), fn=np.array([0, 1]),
               tn=np.array([90, 98]))
    dear = Prices(planned=10.0, unplanned=1000.0, false_alarm=1.0)
    point = choose_operating_point(np.array([1.0, 2.0]), c, dear)
    assert point.threshold == 1.0 and point.index == 0, (
        f"with a missed failure at 1000 and a false alarm at 1, the cheap threshold is 1.00; "
        f"got {point.threshold}. A threshold of 2.00 means you maximised something — "
        "accuracy is 0.99 there and 0.92 at the right answer"
    )
    assert point.counts == Counts(2, 8, 0, 90) and all(
        isinstance(v, int) for v in point.counts), (
        f"the returned counts are the plain ints at the chosen index, got {point.counts!r}"
    )
    assert np.isclose(point.cost, 2 * 10.0 + 0 * 1000.0 + 8 * 1.0), (
        f"cost at the chosen index is 28.0, got {point.cost}"
    )
    assert np.isclose(point.accuracy, 92 / 100), (
        f"accuracy at the chosen index is 0.92, got {point.accuracy}"
    )
    cheap = Prices(planned=10.0, unplanned=12.0, false_alarm=1000.0)
    assert choose_operating_point(np.array([1.0, 2.0]), c, cheap).threshold == 2.0, (
        "re-price a false alarm at 1000 and the same counts choose the other threshold; if "
        "this one did not move, you are not reading `prices`"
    )
    flat = Counts(tp=np.array([1, 1, 1]), fp=np.array([1, 1, 1]), fn=np.array([1, 1, 1]),
                  tn=np.array([1, 1, 1]))
    tie = choose_operating_point(np.array([1.0, 2.0, 3.0]), flat, dear)
    assert tie.threshold == 1.0, (
        f"on a flat cost curve take the LOWEST threshold, got {tie.threshold}. argmin does "
        "this already; argmax of the negative cost does not"
    )
    for bad in ((np.array([]), c), (np.array([1.0, 2.0, 3.0]), c)):
        try:
            choose_operating_point(bad[0], bad[1], dear)
        except ValueError:
            pass
        else:
            raise AssertionError("an empty grid and a length mismatch must both raise "
                                 "ValueError")
    print("exercise 6 looks right — the cost-optimal threshold, ties broken low")

In [ ]:
_try("exercise 6", _check_choose_operating_point)

Now the sweep on this fleet, at the audited prices, and the two comparisons the whole
programme has been building towards. The first is module 1's: what accuracy would have
chosen. The second is the one this capstone adds — what happens if you tune the threshold
on the leaky feature and then deploy that number onto the causal one, which is what
actually happens when a backtest and a gateway are written by different people.

In [ ]:
SWEPT = None
POINT = None
# HEALTH comes from section 4's probe (exercises 1 and 2) and RUNS from section 6's policy
# (exercises 3 and 4), so this cell, and every later one that reads SWEPT or POINT, waits on
# all six exercises rather than on the two or three whose functions it calls.
_FOR_POINT = ("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
              "exercise 6")


def _choose_and_compare() -> None:
    global SWEPT, POINT
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    SWEPT = sweep_counts(HEALTH, RUNS, THRESHOLDS)
    POINT = choose_operating_point(THRESHOLDS, SWEPT, ev.prices)
    acc_index = int(np.argmax(accuracy(SWEPT)))
    acc_counts = Counts(*(int(np.asarray(f)[acc_index]) for f in SWEPT))
    acc_cost = float(expected_cost(acc_counts, ev.prices))

    leaky_swept = sweep_counts(TEMPTING, RUNS, THRESHOLDS)
    leaky_point = choose_operating_point(THRESHOLDS, leaky_swept, ev.prices)
    shipped = alarm_counts(HEALTH, RUNS, leaky_point.threshold)
    shipped_cost = float(expected_cost(shipped, ev.prices))

    print(f"  {'threshold chosen by':<30}{'thr':>7}{'caught':>8}{'missed':>8}{'false':>7}"
          f"{'accuracy':>10}{'cost on the causal feature':>28}")
    print(f"  {'cost (this programme)':<30}{POINT.threshold:>7.2f}{POINT.counts.tp:>8}"
          f"{POINT.counts.fn:>8}{POINT.counts.fp:>7}{POINT.accuracy:>10.4f}"
          f"{POINT.cost:>28,.0f}")
    print(f"  {'accuracy':<30}{THRESHOLDS[acc_index]:>7.2f}{acc_counts.tp:>8}"
          f"{acc_counts.fn:>8}{acc_counts.fp:>7}"
          f"{float(accuracy(acc_counts)):>10.4f}{acc_cost:>28,.0f}")
    print(f"  {'the leaky backtest':<30}{leaky_point.threshold:>7.2f}{shipped.tp:>8}"
          f"{shipped.fn:>8}{shipped.fp:>7}{float(accuracy(shipped)):>10.4f}"
          f"{shipped_cost:>28,.0f}")
    print(f"\naccuracy picks {THRESHOLDS[acc_index]:.2f}, scores "
          f"{acc_counts.tp + acc_counts.tn - POINT.counts.tp - POINT.counts.tn:+d} pumps "
          f"more correctly, gives up {acc_counts.fn - POINT.counts.fn} more failures and "
          f"costs\n{acc_cost / POINT.cost:.2f}x as much. That is module 1, re-measured on "
          f"this asset class and these audited prices.")
    moved = abs(leaky_point.threshold - POINT.threshold)
    print(f"the leaky backtest picks {leaky_point.threshold:.2f}, "
          f"{'the same threshold as yours' if moved == 0 else f'{moved:.2f} away from yours'}"
          f", and promises {leaky_point.cost:,.0f}.")
    print(f"deployed onto the feature the gateway can actually compute, that number costs "
          f"{shipped_cost:,.0f}:\nthe promise was out by "
          f"{100 * abs(shipped_cost / leaky_point.cost - 1):.0f}%, in the "
          f"{'pessimistic' if shipped_cost < leaky_point.cost else 'optimistic'} direction, "
          f"and nothing inside the\nbacktest said which direction it would be. That is the "
          f"objection: not that a leaky score is\nalways flattering, but that it is not a "
          f"forecast of anything.")
    if shipped_cost > POINT.cost:
        print(f"on this fleet it also lands {shipped_cost - POINT.cost:,.0f} above the best "
              f"available ({shipped_cost / POINT.cost:.2f}x).")
    print(f"the probe in section 4 cost one function call and told you this before a single "
          f"threshold was\nswept. It belongs in the specification as a number, next to the "
          f"word 'causal'.")


_try("the operating point", _choose_and_compare, needs=_FOR_POINT)

The same picture, as a curve. The cost minimum and the accuracy maximum are not in the same
place, and nothing about the detector decides which of them you report.

In [ ]:
def _plot_the_cost_curve() -> None:
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    costs = np.asarray(expected_cost(SWEPT, ev.prices), dtype=float)
    accs = np.asarray(accuracy(SWEPT), dtype=float)
    acc_index = int(np.argmax(accs))
    fig, ax = plt.subplots(figsize=(9.5, 3.2))
    ax.plot(THRESHOLDS, costs / 1e6, lw=1.6, color="crimson")
    ax.axvline(POINT.threshold, color="0.3", lw=1.0, ls="--")
    ax.axvline(THRESHOLDS[acc_index], color="0.6", lw=1.0, ls=":")
    ax.annotate(f"cost-optimal {POINT.threshold:.2f}", (POINT.threshold, POINT.cost / 1e6),
                textcoords="offset points", xytext=(8, 12), fontsize=8)
    ax.annotate(f"accuracy-optimal {THRESHOLDS[acc_index]:.2f}",
                (THRESHOLDS[acc_index], costs[acc_index] / 1e6), textcoords="offset points",
                xytext=(-104, -4), fontsize=8)
    twin = ax.twinx()
    twin.plot(THRESHOLDS, accs, lw=1.0, color="0.55")
    twin.set_ylabel("accuracy (grey)", fontsize=9)
    ax.set_xlabel("threshold on the health index")
    ax.set_ylabel("expected cost, millions (red)")
    ax.set_title("one detector, one fleet, two defensible-sounding answers", fontsize=9)
    fig.tight_layout()
    _show(fig)
    print(f"the red minimum is at {POINT.threshold:.2f} and the grey maximum at "
          f"{THRESHOLDS[acc_index]:.2f}. Same counts, same detector,")
    print("different question. Only one of the two questions is asked in the currency the "
          "plant spends.")


_try("the cost curve", _plot_the_cost_curve, needs=_FOR_POINT)

## 9. Exercise 7 — `failures_given_up()`: name them

This is the exercise the capstone exists for. Your threshold does not catch everything, and
a specification that does not say which failures it accepts is not a specification — it is
a number with a confident tone. So: for the threshold you chose, list every failure it does
**not** catch, with the hour the pump broke, the hour it alarmed if it ever did, how much
warning that bought, and which of the two reasons applies.

The two reasons are not interchangeable, and the response to each is different:

- **never alarmed** — the feature never crossed the threshold while the pump was being
  watched. Lowering the threshold would catch it, at a price in false alarms you can compute.
- **alarmed too late** — the feature crossed, but with less than `lead_hours` of warning.
  Lowering the threshold may not help at all: if the defect's whole P-F interval is shorter
  than the lead time the maintenance contract asks for, **no threshold on this feature
  catches it**, and the honest thing to write down is that this asset class has a failure
  mode condition monitoring does not address.

<details><summary>💡 Hint 1 — what to think about</summary>

Which runs can appear in this list at all? A pump that was pulled for a line
reconfiguration did not fail, so it cannot be a failure you gave up. And for a pump that
never alarmed, what is the warning — zero, or something that says "there was none"? A pump
that alarmed exactly `lead_hours` before it broke: caught, or given up?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate the threshold, the lead time and every run's unit against the rows of `health`
first, so a missing pump is your ValueError and not an IndexError from inside
`alarm_hours`. Then call the given `alarm_hours` once at this threshold and walk the
FAILURE runs only. A run with an alarm at least `lead_hours` before its end is caught and
is not in the list. Everything else is: `TOO_LATE` if it alarmed at all, `NEVER_ALARMED` if
it did not, with both hour fields set to the "there was none" value the docstring names.
Return them sorted by unit.
</details>

In [ ]:
def failures_given_up(health: np.ndarray, runs: Sequence, threshold: float,
                      lead_hours: int = LEAD_HOURS) -> tuple:
    """Every failure this threshold does not catch in time, named, with its reason.

    A FAILURE run is caught when it alarmed and `end_hour - alarm_hour >= lead_hours`;
    exactly `lead_hours` of warning counts as caught. Every other FAILURE run appears here:

      * it alarmed, but late  -> `reason=TOO_LATE`, `warning_hours = end_hour - alarm_hour`;
      * it never alarmed      -> `reason=NEVER_ALARMED`, `alarm_hour = warning_hours = -1`.

    CENSORED runs never appear: a pump that did not fail is not a failure you gave up.

    Raise `ValueError` if `threshold` is not finite, if `lead_hours` is negative, or if
    `health` does not have one row per unit named in `runs`.

    Returns: a tuple of `GivenUp`, sorted by unit. Empty when the threshold catches
    everything — which on a real fleet means the threshold is too low, not that you are done.

    Example:
        >>> h = np.array([[1.0, 2.0, 2.0], [1.0, 1.0, 1.0]])
        >>> r = (Run(0, 2, FAILURE), Run(1, 2, FAILURE))
        >>> failures_given_up(h, r, 1.5, lead_hours=2)
        (GivenUp(unit=0, end_hour=2, alarm_hour=1, warning_hours=1, reason='alarmed too late'), GivenUp(unit=1, end_hour=2, alarm_hour=-1, warning_hours=-1, reason='never alarmed'))
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_failures_given_up() -> None:
    h = np.array([[1.0, 2.0, 2.0], [1.0, 1.0, 1.0]])
    r = (Run(0, 2, FAILURE), Run(1, 2, FAILURE))
    got = failures_given_up(h, r, 1.5, lead_hours=2)
    assert got == (GivenUp(0, 2, 1, 1, TOO_LATE), GivenUp(1, 2, -1, -1, NEVER_ALARMED)), (
        f"expected pump 0 late with one hour of warning and pump 1 never alarmed; got {got}"
    )
    edge = failures_given_up(np.array([[1.0, 2.0, 2.0]]), (Run(0, 2, FAILURE),), 1.5,
                             lead_hours=1)
    assert edge == (), (
        f"exactly lead_hours of warning counts as CAUGHT — compare with >=, not > — so this "
        f"pump must not appear; got {edge}"
    )
    censored = failures_given_up(h, (Run(0, 2, CENSORED), Run(1, 2, CENSORED)), 1.5,
                                 lead_hours=2)
    assert censored == (), (
        f"a censored run is not a failure and can never be a failure you gave up; got "
        f"{censored}. A suspension that alarmed is a false alarm, and it is counted as one "
        "by classify_runs — not here"
    )
    after = failures_given_up(np.array([[1.0, 1.0, 9.0]]), (Run(0, 1, FAILURE),), 1.5,
                              lead_hours=0)
    assert after == (GivenUp(0, 1, -1, -1, NEVER_ALARMED),), (
        f"the crossing at hour 2 happened after the pump had already stopped at hour 1, so "
        f"it is not an alarm; got {after}. alarm_hours masks each unit to its own run"
    )
    order = failures_given_up(np.ones((3, 2)), (Run(2, 1, FAILURE), Run(0, 1, FAILURE)),
                              5.0, lead_hours=1)
    assert [g.unit for g in order] == [0, 2], (
        f"the report comes back sorted by unit, got {[g.unit for g in order]}"
    )
    for bad in ({"threshold": np.nan}, {"threshold": np.inf}, {"lead_hours": -1}):
        try:
            failures_given_up(h, r, **{"threshold": 1.5, "lead_hours": 2, **bad})
        except ValueError:
            pass
        else:
            raise AssertionError(f"{bad} must raise ValueError")
    try:
        failures_given_up(h, (Run(7, 2, FAILURE),), 1.5, lead_hours=2)
    except ValueError:
        pass
    else:
        raise AssertionError("a run naming a unit that has no row in health must raise")
    print("exercise 7 looks right — every failure the threshold gives up, with its reason")

In [ ]:
_try("exercise 7", _check_failures_given_up)

Now the list for your own operating point. Read it as the maintenance manager will: these
are pumps, with numbers, that this programme has decided in advance not to save.

In [ ]:
def _name_the_give_ups() -> None:
    given = failures_given_up(HEALTH, RUNS, POINT.threshold)
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"at threshold {POINT.threshold:.2f} this programme catches {POINT.counts.tp} of "
          f"{POINT.counts.tp + POINT.counts.fn} failures and gives up {len(given)}:\n")
    print(f"  {'pump':>6}{'broke at':>10}{'alarmed at':>12}{'warning, h':>12}  reason")
    for g in given:
        print(f"  {g.unit:>6}{g.end_hour:>10}"
              f"{g.alarm_hour if g.alarm_hour >= 0 else '-':>12}"
              f"{g.warning_hours if g.warning_hours >= 0 else '-':>12}  {g.reason}")
    late = [g for g in given if g.reason == TOO_LATE]
    never = [g for g in given if g.reason == NEVER_ALARMED]
    print(f"\n{len(late)} alarmed too late and {len(never)} never alarmed. Those pumps "
          f"account for {len(given) * ev.prices.unplanned:,.0f} of the\n"
          f"{POINT.cost:,.0f} this programme expects to spend — the unplanned price, once "
          f"for each of them.")
    if late:
        spans = [g.warning_hours for g in late]
        window = (f"{spans[0]} hours" if len(set(spans)) == 1
                  else f"between {min(spans)} and {max(spans)} hours")
        print(f"the late {'one' if len(late) == 1 else 'ones'} gave {window} of warning "
              f"against a {LEAD_HOURS} h requirement.")
    # What would it take to catch them? Lower the threshold until each one is caught, and
    # price the false alarms that come with it.
    print(f"\nand here is the price of not giving them up — the highest threshold that "
          f"catches each one,\nand what the whole fleet then costs:\n")
    print(f"  {'to also catch pump':>20}{'threshold':>11}{'false alarms':>14}"
          f"{'total cost':>14}{'vs yours':>11}")
    unreachable = []
    for g in given:
        catches = [t for t in THRESHOLDS
                   if g.unit not in {x.unit for x in failures_given_up(HEALTH, RUNS, t)}]
        if catches:
            t = max(catches)
            counts = alarm_counts(HEALTH, RUNS, t)
            cost = float(expected_cost(counts, ev.prices))
            print(f"  {g.unit:>20}{t:>11.2f}{counts.fp:>14}{cost:>14,.0f}"
                  f"{cost / POINT.cost:>10.2f}x")
        else:
            unreachable.append(g.unit)
            print(f"  {g.unit:>20}{'none':>11}{'-':>14}{'-':>14}{'-':>11}")
    extra = []
    for g in given:
        catches = [t for t in THRESHOLDS
                   if g.unit not in {x.unit for x in failures_given_up(HEALTH, RUNS, t)}]
        if catches:
            counts = alarm_counts(HEALTH, RUNS, max(catches))
            extra.append(float(expected_cost(counts, ev.prices)) - POINT.cost)
    if extra:
        print(f"\nbuying any one of those catches means dropping the threshold into the "
              f"healthy population's\nown noise and alarming on most of the fleet. The "
              f"cheapest of them adds {min(extra):,.0f} to the\nfleet's bill and the "
              f"dearest adds {max(extra):,.0f}, against a failure priced at "
              f"{ev.prices.unplanned:,.0f}.")
        print("that arithmetic — not a shrug about recall — is what the word 'deliberately' "
              "is doing\nin 'the failures it deliberately gives up'.")
    if unreachable:
        print(f"pump(s) {unreachable} cannot be caught at ANY threshold on this grid: the "
              f"whole P-F interval\nis shorter than the {LEAD_HOURS} h the contract asks "
              f"for. That is a failure mode condition\nmonitoring does not address, and the "
              f"specification says so rather than averaging it away.")


_try("the failures given up", _name_the_give_ups, needs=_FOR_POINT + ("exercise 7",))

## 10. Exercise 8 — `deployment_report()`: the constraint it respects

Module 7's gateway does not care how good your feature is. It has an arena of
`GATEWAY.arena_bytes` bytes for the whole monitoring task, it allocates nothing after
start-up, and it publishes and compares the health index in **Q8.8** — one 16-bit word with
eight fractional bits. Two things follow, and both belong in the specification:

- a causal median over `window` hours means keeping `window` readings per pump, for ever.
  Multiply by every pump in the fleet (`N_UNITS`) and the window is a memory decision, not a filter-design decision;
- the threshold you publish is not the threshold that runs. Q8.8 can hold only multiples
  of 1/256, so `1.25` survives the trip exactly and `1.40` and `1.15` do not. The gateway
  compares the nearest representable numbers, and the
  disagreement between your float arithmetic and its fixed-point arithmetic is a count you
  can measure rather than a risk you can describe.

<details><summary>💡 Hint 1 — what to think about</summary>

What does the gateway have to keep per pump, and for how long? And when you round a health
index and a threshold to the nearest 1/256, how many readings change side — and how many of
those changes actually move a pump's first alarm rather than a reading in the middle of a
quiet week?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

A pump costs the gateway its retained window of readings plus its per-pump constants; the
fleet costs that for every row of `health`, and it fits when the total is no more than the
arena. On the wire, the health index AND the threshold each become the NEAREST step the
format can hold — rounding in both directions, never truncating, because a threshold
rounded down alarms earlier than the published number. Then count two different things:
the readings whose alarm verdict differs between float and wire, and the pumps whose FIRST
crossing differs, with "never crosses" as a value of its own.
</details>

In [ ]:
def deployment_report(health: np.ndarray, threshold: float, window: int = WINDOW,
                      limits: GatewayLimits = GATEWAY) -> DeploymentReport:
    """Does this programme fit the gateway, and does it mean the same thing there?

    * `bytes_per_unit = window * limits.sample_bytes + limits.constant_bytes`;
    * `total_bytes = bytes_per_unit * (number of rows in health)`;
    * `fits = total_bytes <= limits.arena_bytes`;
    * `wire_threshold`: `threshold` rounded to the nearest multiple of
      `1 / 2 ** limits.wire_frac` — what the gateway will actually compare against;
    * `disagreeing_readings`: readings where `health >= threshold` and the Q-rounded
      `health >= wire_threshold` disagree;
    * `units_whose_alarm_moves`: rows whose FIRST crossing index differs between the two,
      counting "never crosses" as a value of its own.

    Raise `ValueError` if `health` is not 2-D, if `threshold` is not finite, if `window < 1`,
    if any of `arena_bytes`, `sample_bytes` or `constant_bytes` is not positive, or if
    `wire_frac` is outside `1 .. 30`.

    Returns: a `DeploymentReport`.

    Example, two pumps, a four-hour window and a tiny arena:
        >>> lim = GatewayLimits(arena_bytes=40, sample_bytes=4, constant_bytes=12, wire_frac=8)
        >>> deployment_report(np.array([[1.0, 2.0], [1.0, 1.0]]), 1.5, window=4, limits=lim)
        DeploymentReport(bytes_per_unit=28, total_bytes=56, fits=False, wire_threshold=1.5, disagreeing_readings=0, units_whose_alarm_moves=0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_deployment_report() -> None:
    lim = GatewayLimits(arena_bytes=40, sample_bytes=4, constant_bytes=12, wire_frac=8)
    got = deployment_report(np.array([[1.0, 2.0], [1.0, 1.0]]), 1.5, window=4, limits=lim)
    assert got.bytes_per_unit == 28 and got.total_bytes == 56, (
        f"4 hours x 4 bytes + 12 bytes of constants = 28 per pump, 56 for two; got "
        f"{got.bytes_per_unit} and {got.total_bytes}"
    )
    assert got.fits is False, (
        "56 bytes do not fit in a 40-byte arena; `fits` is a bool, and it is the whole "
        "point of the report"
    )
    big = deployment_report(np.zeros((2, 2)), 1.5, window=4,
                            limits=lim._replace(arena_bytes=56))
    assert big.fits is True, "exactly filling the arena fits; compare with <=, not <"
    assert np.isclose(got.wire_threshold, 1.5), (
        f"1.5 is exactly representable in Q8.8, so it comes back unchanged; got "
        f"{got.wire_threshold}"
    )
    coarse = GatewayLimits(arena_bytes=10 ** 6, sample_bytes=4, constant_bytes=12, wire_frac=1)
    half = deployment_report(np.array([[1.30, 1.26]]), 1.30, window=1, limits=coarse)
    assert np.isclose(half.wire_threshold, 1.5), (
        f"with one fractional bit the grid is 0.5 wide, so 1.30 rounds to 1.5; got "
        f"{half.wire_threshold}. Rounding the threshold DOWN would quietly make the gateway "
        "alarm earlier than the number you published"
    )
    assert half.disagreeing_readings == 1, (
        f"on the wire both readings round to 1.5 and both clear the 1.5 threshold, while in "
        f"float only the first clears 1.30 — one reading disagrees, got "
        f"{half.disagreeing_readings}. Rounding the health index but comparing it with the "
        "unrounded threshold, or the other way round, misses this"
    )
    assert half.units_whose_alarm_moves == 0, (
        f"both crossings still start at hour 0, so no pump's first alarm moved; got "
        f"{half.units_whose_alarm_moves}. A reading that changes side in the middle of a "
        "quiet week is not an alarm that moved"
    )
    moved = deployment_report(np.array([[1.26, 1.30]]), 1.30, window=1, limits=coarse)
    assert moved.units_whose_alarm_moves == 1, (
        f"in float this pump first crosses at hour 1; on the wire both readings round to 1.5 "
        f"and it crosses at hour 0, so its first alarm moved an hour EARLIER than the "
        f"published threshold implies; got {moved.units_whose_alarm_moves}"
    )
    # The threshold rounds DOWN on this grid and a reading rounds UP to meet it: on the wire
    # the gateway alarms on a pump the published threshold would have left alone.
    both_ways = deployment_report(np.array([[1.10]]), 1.20, window=1, limits=coarse)
    assert both_ways.disagreeing_readings == 1, (
        f"1.20 becomes 1.0 on this grid and 1.10 becomes 1.0 too, so the wire alarms where "
        f"the float does not: one reading disagrees, got {both_ways.disagreeing_readings}. "
        "A zero here means you compared the rounded health index against the UNROUNDED "
        "threshold, which is not what the gateway does"
    )
    assert both_ways.units_whose_alarm_moves == 1, (
        f"and that pump never alarms in float and alarms at hour 0 on the wire, so its first "
        f"alarm moved; got {both_ways.units_whose_alarm_moves}"
    )
    never = deployment_report(np.array([[1.0, 1.0]]), 9.0, window=1, limits=coarse)
    assert never.units_whose_alarm_moves == 0 and never.disagreeing_readings == 0, (
        f"a pump that never crosses in either arithmetic has not moved; got {never}"
    )
    for bad_health, bad_kwargs in ((np.zeros(4), {}), (np.zeros((2, 2)), {"threshold": np.nan}),
                                  (np.zeros((2, 2)), {"window": 0}),
                                  (np.zeros((2, 2)), {"limits": lim._replace(arena_bytes=0)}),
                                  (np.zeros((2, 2)), {"limits": lim._replace(sample_bytes=0)}),
                                  (np.zeros((2, 2)), {"limits": lim._replace(wire_frac=0)}),
                                  (np.zeros((2, 2)), {"limits": lim._replace(wire_frac=31)})):
        try:
            deployment_report(bad_health, **{"threshold": 1.5, "window": 4, "limits": lim,
                                             **bad_kwargs})
        except ValueError:
            pass
        else:
            raise AssertionError(f"deployment_report with {bad_kwargs!r} must raise ValueError")
    print("exercise 8 looks right — the arena budget and the Q-format disagreement")

In [ ]:
_try("exercise 8", _check_deployment_report)

The window is the decision this makes visible. Offline it is a filter length; on the
gateway it is `N_UNITS` pumps' worth of retained readings. Run the sweep: for each window, build
the feature, re-derive the cost-optimal threshold on it, and ask the gateway whether it
will hold.

In [ ]:
def _sweep_the_window() -> None:
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    print(f"  {'window, h':>10}{'bytes/pump':>12}{'total bytes':>13}{'fits':>7}"
          f"{'threshold':>11}{'missed':>8}{'false':>7}{'cost':>14}")
    best_any = best_fitting = None
    for w in (3, 5, 9, 11, 17, 25, 49):
        feature = causal_health(RAW, DUTY, AMBIENT, BASELINE_HOURS, w)
        point = choose_operating_point(THRESHOLDS, sweep_counts(feature, RUNS, THRESHOLDS),
                                       ev.prices)
        rep = deployment_report(feature, point.threshold, w, GATEWAY)
        print(f"  {w:>10}{rep.bytes_per_unit:>12}{rep.total_bytes:>13,}"
              f"{('yes' if rep.fits else 'NO'):>7}{point.threshold:>11.2f}"
              f"{point.counts.fn:>8}{point.counts.fp:>7}{point.cost:>14,.0f}")
        if best_any is None or point.cost < best_any[1]:
            best_any = (w, point.cost)
        if rep.fits and (best_fitting is None or point.cost < best_fitting[1]):
            best_fitting = (w, point.cost)
    arena = GATEWAY.arena_bytes
    widest = (arena // N_UNITS - GATEWAY.constant_bytes) // GATEWAY.sample_bytes
    print(f"\nthe arena is {arena:,} bytes for {N_UNITS} pumps, so the widest window that "
          f"fits is {widest} hours.")
    if best_any[0] == best_fitting[0]:
        print(f"on this fleet the cheapest window, {best_any[0]} h, is also one that fits. "
              f"The constraint did not\nbite here — and it was still worth checking, because "
              f"nothing about the offline score would\nhave told you.")
    else:
        print(f"the cheapest window overall is {best_any[0]} h at {best_any[1]:,.0f}, and it "
              f"does not fit.\nthe cheapest window that DOES fit is {best_fitting[0]} h at "
              f"{best_fitting[1]:,.0f} — "
              f"{best_fitting[1] - best_any[1]:,.0f} more,\nwhich is what this gateway costs "
              f"per {N_HOURS} h, stated as a number instead of as a shrug.")
    rep = deployment_report(HEALTH, POINT.threshold, WINDOW, GATEWAY)
    moves = rep.units_whose_alarm_moves
    print(f"\nthe programme ships at window {WINDOW} h — the widest that fits — and "
          f"threshold {POINT.threshold:.2f}. On the\nwire that threshold is "
          f"{rep.wire_threshold:.6f}, {rep.disagreeing_readings} of {HEALTH.size:,} readings "
          f"change side, and the first alarm\nmoves on {moves} "
          f"{'pump' if moves == 1 else 'pumps'} out of {N_UNITS}. Small, measured, and "
          f"written down — which is the difference\nbetween a deployment constraint you "
          f"respect and one you have not looked at.")


_try("the deployment constraint", _sweep_the_window, needs=_FOR_POINT + ("exercise 8",))

## 11. Exercise 9 — `rederivation_trigger()`: when this number stops being true

Module 8 ended with a loop and an instruction: re-derive the threshold whenever a price is
re-measured or the feature distribution moves. A specification that writes that down as a
sentence has written down a good intention. Write it down as **two numbers** and it becomes
a trigger somebody can put in a calendar:

- **how far a price has to move** before the threshold does. Walk a grid of multipliers on
  one price, re-derive at each, and report the smallest relative move that changes the
  answer. Under it, nothing to do; over it, this document is out of date;
- **the drift alert level**, which module 8 measured on this plant rather than quoting from
  a rule of thumb. `null_psi_band` is given; the trigger just carries it.

<details><summary>💡 Hint 1 — what to think about</summary>

Some multipliers move the threshold and some do not. Of the ones that do, which is the one
you report — and what do you report when none of them does? Is a 10% rise in a price the
same size of move as a 10% fall? And `which` names one of three prices: does your loop
scale the price it names, or always the one you had in mind?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate `which`, the multipliers and `drift_alert` first. Choose once at the given prices
to get the threshold in force. Then for every multiplier, rebuild the prices with that one
field scaled and choose again; whenever the threshold differs and `abs(m - 1)` is strictly
smaller than the smallest seen so far, keep it and the threshold it moved to. If nothing in
the grid moves it, report the "not within the range we looked at" value the docstring
names — never zero — and the unchanged threshold.
</details>

In [ ]:
def rederivation_trigger(thresholds: np.ndarray, counts: Counts, prices: Prices,
                         multipliers: np.ndarray = PRICE_MULTIPLIERS,
                         drift_alert: float = 0.0, which: str = "unplanned") -> Trigger:
    """How far one price can move before this threshold is the wrong threshold.

    `which` names the field of `prices` to scan and must be one of `OUTCOMES`. For each
    multiplier `m`, re-derive the cost-optimal threshold with that field multiplied by `m`.
    `price_move_fraction` is the SMALLEST `abs(m - 1)` at which the threshold differs from
    the one in force, and `moved_threshold` is what it moves to there. If no multiplier in
    the grid moves it, report `float("inf")` and the unchanged threshold — the honest answer
    is "not within the range we looked at", never "never".

    Raise `ValueError` if `which` is not in `OUTCOMES`, if `multipliers` is empty or holds a
    value that is not strictly positive, or if `drift_alert` is negative or not finite.

    Returns: a `Trigger`.

    Example, a two-point grid where halving the unplanned price flips the answer:
        >>> c = Counts(tp=np.array([2, 1]), fp=np.array([8, 0]),
        ...            fn=np.array([0, 1]), tn=np.array([90, 98]))
        >>> t = rederivation_trigger(np.array([1.0, 2.0]), c,
        ...                          Prices(10.0, 1000.0, 1.0), np.array([1.0, 0.01]), 0.5)
        >>> t.threshold, t.price_move_fraction, t.moved_threshold, t.drift_alert
        (1.0, 0.99, 2.0, 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rederivation_trigger() -> None:
    c = Counts(tp=np.array([2, 1]), fp=np.array([8, 0]), fn=np.array([0, 1]),
               tn=np.array([90, 98]))
    grid = np.array([1.0, 2.0])
    prices = Prices(10.0, 1000.0, 1.0)
    got = rederivation_trigger(np.array([1.0, 2.0]), c, prices, np.array([1.0, 0.01]), 0.5)
    assert got.threshold == 1.0 and got.moved_threshold == 2.0, (
        f"at these prices 1.00 is in force and a near-zero unplanned price moves it to 2.00; "
        f"got {got.threshold} and {got.moved_threshold}"
    )
    assert np.isclose(got.price_move_fraction, 0.99), (
        f"the move is reported as abs(m - 1) = 0.99, not as the multiplier 0.01 and not as "
        f"a percentage; got {got.price_move_fraction}"
    )
    assert got.drift_alert == 0.5, "drift_alert is carried through unchanged"
    # A grid where a BIG move is met first and a smaller one later: 0.50 flips this
    # threshold and so does 0.90, and 0.90 is the answer.
    close = Counts(tp=np.array([2, 1]), fp=np.array([20, 2]), fn=np.array([1, 2]),
                   tn=np.array([77, 95]))
    nearest = rederivation_trigger(np.array([1.0, 2.0]), close, Prices(10.0, 30.0, 1.0),
                                   np.array([1.0, 0.5, 0.95, 0.9]), 0.0)
    assert np.isclose(nearest.price_move_fraction, 0.1), (
        f"two multipliers move this threshold, 0.50 and 0.90, and the SMALLEST move — 0.10 — "
        f"is the answer, not the first one met in the grid; got "
        f"{nearest.price_move_fraction}"
    )
    assert nearest.moved_threshold == 2.0, (
        f"moved_threshold is what the threshold becomes at the SMALLEST moving multiplier; "
        f"got {nearest.moved_threshold}"
    )
    stuck = rederivation_trigger(np.array([1.0, 2.0]), c, prices, np.array([1.0, 1.05]), 0.0)
    assert stuck.price_move_fraction == float("inf") and stuck.moved_threshold == 1.0, (
        f"no multiplier in that grid moves the threshold, so the honest report is inf and "
        f"the unchanged threshold; got {stuck.price_move_fraction} and "
        f"{stuck.moved_threshold}. Returning 0.0 would say the threshold is about to move"
    )
    fa = rederivation_trigger(np.array([1.0, 2.0]), c, prices, np.array([1.0, 500.0]), 0.0,
                              which="false_alarm")
    assert fa.moved_threshold == 2.0, (
        f"`which` has to be read: scaling the FALSE ALARM price by 500 moves this threshold "
        f"to 2.00, got {fa.moved_threshold}. An implementation that always scales `unplanned` "
        "passes every other case here"
    )
    for bad in ({"which": "unplanned_repair"}, {"multipliers": np.array([])},
                {"multipliers": np.array([1.0, 0.0])},
                {"multipliers": np.array([1.0, -1.0])},
                {"drift_alert": -0.1}, {"drift_alert": np.nan}):
        try:
            rederivation_trigger(np.array([1.0, 2.0]), c, prices,
                                 **{"multipliers": grid, "drift_alert": 0.0, **bad})
        except ValueError:
            pass
        else:
            raise AssertionError(f"rederivation_trigger with {bad!r} must raise ValueError")
    print("exercise 9 looks right — the price move that invalidates this threshold")

In [ ]:
# Re-deriving a threshold means choosing it again, so this check runs your
# choose_operating_point too. It waits for exercise 6 to pass: a wrong chooser would
# otherwise fail THIS exercise, and send you to debug code that may be right.
_try("exercise 9", _check_rederivation_trigger, needs=("exercise 6",))

Run it on this fleet, for all three prices, with the drift alert level measured on the
commissioning block. The column to look at is the last one: it is the sentence the
specification will carry.

In [ ]:
PSI_ALERT = None


def _show_the_triggers() -> None:
    global PSI_ALERT
    ev = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    PSI_ALERT = null_psi_band(HEALTH[:, :BASELINE_HOURS])
    print(f"  {'price':<14}{'now':>11}{'provenance':>11}{'threshold moves at':>21}"
          f"{'i.e. at a price of':>30}{'to':>7}")
    for name, prov in zip(OUTCOMES, ev.provenance):
        trig = rederivation_trigger(THRESHOLDS, SWEPT, ev.prices, PRICE_MULTIPLIERS,
                                    PSI_ALERT, which=name)
        here = getattr(ev.prices, name)
        if np.isfinite(trig.price_move_fraction):
            at = f"a move of {100 * trig.price_move_fraction:.0f}%"
            span = (f"{here * (1 - trig.price_move_fraction):,.0f} or "
                    f"{here * (1 + trig.price_move_fraction):,.0f}")
            to = f"{trig.moved_threshold:.2f}"
        else:
            at = f"nothing within {100 * (PRICE_MULTIPLIERS.max() - 1):.0f}%"
            span = "-"
            to = "-"
        print(f"  {name:<14}{here:>11,.0f}{prov:>11}{at:>21}{span:>30}{to:>7}")
    print(f"\ndrift alert: PSI above {PSI_ALERT:.5f} on the health index over {N_BINS} "
          f"commissioning-quantile bins.")
    print(f"that number is this plant's own null band, measured over {NULL_DRAWS} random "
          f"half-splits of the\ncommissioning block — not a rule of thumb, for the same "
          f"reason the threshold is not one.")


_try("the re-derivation trigger", _show_the_triggers, needs=_FOR_POINT + ("exercise 9",))

## 12. The evidence bundle

Everything the runnable half measured, in one object. The specification is checked against
**this** and not against the author's good intentions: a number that appears in the
document and not in the bundle did not come from anywhere.

In [ ]:
_EVIDENCE: list = []


def build_evidence() -> Evidence:
    """Assemble what your own functions measured. Given to you — it calls your code."""
    prices = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    runs = label_runs(ORDERS)
    health = causal_health(RAW, DUTY, AMBIENT)
    swept = sweep_counts(health, runs, THRESHOLDS)
    point = choose_operating_point(THRESHOLDS, swept, prices.prices)
    alert = null_psi_band(health[:, :BASELINE_HOURS])
    return Evidence(
        causal_margin=causality_margin(causal_health, RAW, DUTY, AMBIENT),
        censoring=censoring_report(runs),
        lead_hours=LEAD_HOURS,
        point=point,
        prices=prices,
        given_up=failures_given_up(health, runs, point.threshold),
        deployment=deployment_report(health, point.threshold, WINDOW, GATEWAY),
        trigger=rederivation_trigger(THRESHOLDS, swept, prices.prices, PRICE_MULTIPLIERS,
                                     alert, which="unplanned"),
    )


def evidence() -> Evidence:
    """The bundle, built once and cached. Given to you."""
    if not _EVIDENCE:
        _EVIDENCE.append(build_evidence())
    return _EVIDENCE[0]


def same_number(a: float, b: float, tolerance: float = 1e-9) -> bool:
    """Equality for numbers that may legitimately be infinite. Given to you.

    `rederivation_trigger` reports `inf` when no price move in the grid shifts the threshold,
    and `abs(inf - inf)` is nan rather than zero, so a plain tolerance test would call two
    identical answers different.
    """
    if np.isinf(a) or np.isinf(b):
        return bool(a == b)
    return bool(abs(float(a) - float(b)) <= float(tolerance))


def _show_evidence() -> None:
    ev = evidence()
    print(f"  causal margin        {ev.causal_margin:.10f}")
    print(f"  censors              {ev.censoring.n_censored} runs, "
          f"{ev.censoring.unobserved_unit_hours:,} unobserved unit-hours")
    print(f"  lead requirement     {ev.lead_hours} h")
    print(f"  threshold            {ev.point.threshold:.2f} at {ev.point.cost:,.0f}")
    print(f"  prices               " + " · ".join(
        f"{n} {v:,.0f} ({p})" for n, v, p in zip(OUTCOMES, ev.prices.prices,
                                                 ev.prices.provenance)))
    print(f"  gives up             pumps {list(g.unit for g in ev.given_up)}")
    print(f"  deployment           {ev.deployment.total_bytes:,} bytes, fits="
          f"{ev.deployment.fits}, wire threshold {ev.deployment.wire_threshold:.6f}")
    print(f"  re-derive            price move "
          f"{100 * ev.trigger.price_move_fraction:.0f}% · drift PSI "
          f"{ev.trigger.drift_alert:.5f}")


_try("the evidence bundle", _show_evidence, needs=_EVIDENCE_NEEDS)

## 13. Exercise 10 — `check_specification()`: the checklist you run before you submit

This is the checker, and it is the second half of the deliverable. Eight items, each one
comparing a line of the document with the evidence bundle, each returning a `CheckResult`
whose `detail` says what is wrong in words the author can act on. `specification_passes`
is given and requires **all eight**: there is no partial credit on a specification.

Two of the eight are the pass condition this programme has been arguing for since module 1,
and they are the two a good model tempts you to skip:

- **the three prices carry their provenance.** A threshold reported without the prices it
  came from, or with prices whose evidence is not stated, fails. It does not matter how
  good the detector is.
- **the failures given up are named.** A specification whose `gives_up` is empty while the
  evidence names two pumps fails. So does one that names the wrong pumps.

<details><summary>💡 Hint 1 — what to think about</summary>

For each item, what exactly makes it true — a sentence being present, a number matching the
evidence, or both? Agreement is not always enough: a document can faithfully report a
feature that leaks, a lead time of zero, or a budget the gateway cannot hold. And for the
two items this module exists for: what happens to an empty `gives_up` when the evidence
names pumps, or to a provenance line the audit does not support?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate the two arguments first. Then build the results one at a time, in `CHECKLIST_ITEMS`
order, appending a `CheckResult` for each: compare the prose fields for being non-empty
after stripping, compare the tuples for exact equality, and compare the floats — some of
which may be infinite — with `same_number` at the given tolerance. Each item reads only its
own fields, so breaking one line fails one item and leaves the other seven alone. Write the
`detail` as if the reader has to fix it — name the value found and the value expected,
never "mismatch".
</details>

In [ ]:
def check_specification(spec: Specification, evidence_bundle: Evidence,
                        tolerance: float = 1e-9) -> tuple:
    """Run the eight-item submission checklist over one specification.

    Returns one `CheckResult` per entry of `CHECKLIST_ITEMS`, in that order, where `ok` is:

      1. `feature` is a non-empty string, `causal_margin` matches the evidence, AND the
         evidence's margin is exactly 0.0 — a feature measured as leaking is not causal
         however carefully the document describes it;
      2. `labelling_policy` is a non-empty string and `censors_units` equals the evidence's
         censored units exactly;
      3. `lead_hours` equals the evidence's lead hours and is greater than zero;
      4. every field of `prices` matches the evidence's prices and `price_provenance` equals
         the evidence's provenance exactly;
      5. `threshold` matches the evidence's chosen threshold;
      6. `gives_up` equals the units of the evidence's given-up failures, in order;
      7. `deployment_constraint` is a non-empty string, `memory_bytes` matches the evidence,
         and the evidence says it fits;
      8. `rederive_when` holds at least two non-empty strings, and `price_move_fraction` and
         `drift_alert` both match the evidence.

    Numbers are compared with `same_number(..., tolerance)`. `detail` is never empty: on a
    pass it states what was checked, and on a failure it names the value found and the value
    the evidence holds.

    Raise `ValueError` if `spec` is not a `Specification`, if `evidence_bundle` is not an
    `Evidence`, or if `tolerance` is negative or not finite.

    Returns: a tuple of exactly `len(CHECKLIST_ITEMS)` `CheckResult`s.

    Example:
        >>> results = check_specification(my_specification(evidence()), evidence())
        >>> [r.ok for r in results]
        [True, True, True, True, True, True, True, True]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def toy_pair() -> tuple:
    """A small, internally consistent (specification, evidence) pair. Given to you."""
    ev = Evidence(
        causal_margin=0.0,
        censoring=CensoringReport(2, 3, (1, 4, 7), 0.6, 120),
        lead_hours=24,
        point=OperatingPoint(3, 1.75, 1234.0, 0.9, Counts(2, 1, 1, 6)),
        prices=PriceEvidence(Prices(10.0, 100.0, 5.0), (9, 7, 2),
                             (AUDITED, AUDITED, GUESSED)),
        given_up=(GivenUp(4, 90, 88, 2, TOO_LATE), GivenUp(9, 50, -1, -1, NEVER_ALARMED)),
        deployment=DeploymentReport(56, 560, True, 1.75, 3, 1),
        trigger=Trigger(1.75, 0.2, 2.10, 0.033),
    )
    spec = Specification(
        asset_class="toy pumps", feature="conditioned RMS over a causal median",
        causal_margin=0.0, labelling_policy="work order raised hour, suspensions censored",
        censors_units=(1, 4, 7), lead_hours=24, threshold=1.75,
        prices=Prices(10.0, 100.0, 5.0), price_provenance=(AUDITED, AUDITED, GUESSED),
        gives_up=(4, 9), deployment_constraint="8 KiB arena, Q8.8 on the wire",
        memory_bytes=560, rederive_when=TRIGGERS, price_move_fraction=0.2,
        drift_alert=0.033)
    return spec, ev


def _check_check_specification() -> None:
    spec, ev = toy_pair()
    good = check_specification(spec, ev)
    assert len(good) == len(CHECKLIST_ITEMS), (
        f"one result per checklist item: expected {len(CHECKLIST_ITEMS)}, got {len(good)}"
    )
    assert tuple(r.item for r in good) == CHECKLIST_ITEMS, (
        f"the results come back in CHECKLIST_ITEMS order, got {[r.item for r in good]}"
    )
    assert all(r.ok for r in good), (
        "this specification agrees with its evidence on every line and must pass all eight: "
        f"{[(r.item, r.detail) for r in good if not r.ok]}"
    )
    assert all(isinstance(r.detail, str) and r.detail.strip() for r in good), (
        "every result carries a detail, including the passing ones — the checklist is read "
        "by a person"
    )
    assert specification_passes(good), "specification_passes must agree with eight ok results"

    def fails_only(field: str, value, index: int) -> None:
        broken = check_specification(spec._replace(**{field: value}), ev)
        bad = [i for i, r in enumerate(broken) if not r.ok]
        assert bad == [index], (
            f"setting {field}={value!r} must fail item {index} "
            f"({CHECKLIST_ITEMS[index]!r}) and nothing else; instead the failing items were "
            f"{[CHECKLIST_ITEMS[i] for i in bad]}"
        )
        assert not specification_passes(broken), (
            f"specification_passes must be False when item {index} fails"
        )

    fails_only("feature", "   ", 0)
    fails_only("causal_margin", 0.5, 0)
    fails_only("labelling_policy", "", 1)
    fails_only("censors_units", (1, 4), 1)
    fails_only("lead_hours", 12, 2)
    fails_only("prices", Prices(10.0, 90.0, 5.0), 3)
    fails_only("prices", Prices(100.0, 10.0, 5.0), 3)     # the same three, in the wrong order
    fails_only("price_provenance", (AUDITED, AUDITED, AUDITED), 3)
    fails_only("threshold", 2.0, 4)
    fails_only("gives_up", (), 5)
    fails_only("gives_up", (4, 9, 11), 5)
    fails_only("deployment_constraint", "", 6)
    fails_only("memory_bytes", 561, 6)
    fails_only("rederive_when", ("only one",), 7)
    fails_only("rederive_when", ("a", "  "), 7)
    fails_only("price_move_fraction", 0.25, 7)
    fails_only("drift_alert", 0.04, 7)

    leaky = check_specification(spec._replace(causal_margin=0.5),
                                ev._replace(causal_margin=0.5))
    assert not leaky[0].ok, (
        "a document that honestly reports a non-zero causality margin still fails item 1: "
        "the feature leaks, and reporting the leak accurately does not make it deployable"
    )
    wont_fit = check_specification(spec, ev._replace(
        deployment=ev.deployment._replace(fits=False)))
    assert not wont_fit[6].ok, (
        "a programme that does not fit the gateway fails item 7 even when the document's "
        "byte count is right"
    )
    inf_trigger = ev._replace(trigger=ev.trigger._replace(price_move_fraction=float("inf")))
    assert check_specification(spec._replace(price_move_fraction=float("inf")),
                               inf_trigger)[7].ok, (
        "when no price move in the grid shifts the threshold both sides are inf and the item "
        "passes; use same_number, because abs(inf - inf) is nan"
    )
    for bad in (("spec", ev, ev), ("evidence", spec, spec)):
        try:
            check_specification(bad[1], bad[2])
        except ValueError:
            pass
        else:
            raise AssertionError(f"a wrong type for {bad[0]} must raise ValueError")
    for bad_tol in (-1e-9, np.nan, np.inf):
        try:
            check_specification(spec, ev, tolerance=bad_tol)
        except ValueError:
            pass
        else:
            raise AssertionError(f"tolerance={bad_tol} must raise ValueError")
    print("exercise 10 looks right — eight items, each failing on its own line")

In [ ]:
_try("exercise 10", _check_check_specification)

## 14. Your specification

The one-page document. Fill it in from the **evidence bundle**, not from memory and not
from the notebook's printed output: every number in it must be the number your own code
computed, because that is the only kind of number a reader can check. The three prose
fields are yours to write — describe the feature, the labelling policy and the deployment
constraint in a sentence each, well enough that somebody who has never seen this notebook
could rebuild them.

Then run the checklist. It is the same code your reviewer will run.

<details><summary>💡 Hint 1 — what to think about</summary>

Which fields are numbers or tuples the bundle already holds, and which are sentences only
you can write? The checklist compares `gives_up` with the evidence's give-ups pump by pump,
in order: is each entry of `bundle.given_up` the thing the document asks for, or a record
that contains it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Read every numeric and tuple field off the matching part of `bundle`: the causal margin,
the censoring report's units, the lead time, the operating point's threshold, the price
evidence's prices and provenance, the unit of each given-up record, the deployment
report's total bytes, and the trigger's price move and drift alert. Write the feature, the
labelling policy and the deployment constraint as one plain sentence each, and use the two
`TRIGGERS` from the setup cell. If the checklist still refuses it, each FAIL line names the
field, what the document said and what the evidence measured.
</details>

In [ ]:
def my_specification(bundle: Evidence) -> Specification:
    """YOUR one-page specification for this asset class.

    Every numeric field comes from `bundle`; every prose field is a sentence you write. The
    checklist in exercise 10 compares the two, so a number typed by hand that happens to
    agree is fine and a number typed by hand that does not is the failure this whole module
    is about.

    Returns: a `Specification` that passes `check_specification(spec, bundle)` on all eight
    items.

    Worked example of the shape — one field from the bundle and one written by you:
        >>> Specification(asset_class="toy", feature="what you built", causal_margin=0.0,
        ...               labelling_policy="raised hour", censors_units=(), lead_hours=24,
        ...               threshold=1.0, prices=Prices(1.0, 2.0, 3.0),
        ...               price_provenance=(AUDITED, AUDITED, GUESSED), gives_up=(),
        ...               deployment_constraint="8 KiB", memory_bytes=0,
        ...               rederive_when=TRIGGERS, price_move_fraction=0.1,
        ...               drift_alert=0.0).lead_hours
        24
    """
    # YOUR CODE HERE
    raise NotImplementedError


_MARK = {True: "PASS", False: "FAIL"}


def print_checklist(results: Sequence, title: str = "") -> None:
    """Print one checklist run. Given to you — this is what you look at before submitting."""
    if title:
        print(title)
    for r in results:
        print(f"  [{_MARK[bool(r.ok)]}] {r.item:<42} {r.detail}")
    print(f"  -> {'ACCEPTED' if specification_passes(results) else 'REJECTED'}\n")


def _run_my_checklist() -> None:
    bundle = evidence()
    results = check_specification(my_specification(bundle), bundle)
    print_checklist(results, "YOUR SUBMISSION")
    # The eight PASS/FAIL lines grade the DOCUMENT. This exercise is done only when the
    # document is accepted, so a refusal is a failed check, not a pass that printed REJECTED.
    refused = [r.item for r in results if not r.ok]
    assert specification_passes(results), (
        f"the checklist refuses your specification on {len(refused)} of {len(results)} "
        f"items ({'; '.join(refused) or 'see above'}). Each FAIL line above names what the "
        "document said and what the evidence measured: read that field from the bundle, "
        "not from memory"
    )


_try("your specification", _run_my_checklist, needs=_EVIDENCE_NEEDS + ("exercise 10",))

## 15. What the checklist refuses

Three specifications that a competent person would submit, each defensible until it meets
the evidence. The third is the one the whole programme exists to refuse: the model is
identical, the threshold is identical, and it is still not a monitoring programme.

In [ ]:
def _show_what_is_refused() -> None:
    bundle = evidence()
    mine = my_specification(bundle)
    kickoff_provenance = tuple(GUESSED for _ in OUTCOMES)
    cases = (
        ("A · the threshold, reported without its evidence",
         mine._replace(prices=KICKOFF, price_provenance=kickoff_provenance)),
        ("B · the threshold, with nothing named as given up",
         mine._replace(gives_up=())),
        ("C · a round number everyone was comfortable with",
         mine._replace(threshold=2.00)),
    )
    for title, spec in cases:
        print_checklist(check_specification(spec, bundle), title)
    print("A fails on one line and passes seven. It has a feature, a labelling policy, a "
          "lead time,\na deployment budget and a trigger — and it prices a missed failure at "
          f"{KICKOFF.unplanned:,.0f} because\nthat is what the room guessed, when the plant's "
          f"own invoices say {bundle.prices.prices.unplanned:,.0f}.")
    print(f"B fails because it gives up pumps "
          f"{[g.unit for g in bundle.given_up]} and does not say so. Nothing in it is false. "
          f"It is\nthe specification almost everybody writes, and the maintenance manager "
          f"finds out which\npumps were on the list when one of them breaks.")
    print(f"C ships {2.00:.2f} instead of {bundle.point.threshold:.2f}. On this fleet that "
          f"is a defensible-looking number\nnobody derived, and the checklist can tell, "
          f"because the evidence is in the same room as\nthe document.")


_try("what the checklist refuses", _show_what_is_refused,
     needs=_EVIDENCE_NEEDS + ("exercise 10", "your specification"))

And one more, because it is the failure this programme opened with: a **different
detector**, documented just as carefully. Everything about it is honest except that it
cannot be computed at the hour it claims to be computed, and the checklist refuses it on
line one without looking at its score at all.

In [ ]:
def _refuse_the_other_detector() -> None:
    prices = price_the_outcomes(AUDIT, KICKOFF, MIN_OBSERVATIONS)
    runs = label_runs(ORDERS)
    swept = sweep_counts(TEMPTING, runs, THRESHOLDS)
    point = choose_operating_point(THRESHOLDS, swept, prices.prices)
    leaky_bundle = Evidence(
        causal_margin=causality_margin(tempting_health, RAW, DUTY, AMBIENT),
        censoring=censoring_report(runs), lead_hours=LEAD_HOURS, point=point, prices=prices,
        given_up=failures_given_up(TEMPTING, runs, point.threshold),
        deployment=deployment_report(TEMPTING, point.threshold, WINDOW, GATEWAY),
        trigger=rederivation_trigger(THRESHOLDS, swept, prices.prices, PRICE_MULTIPLIERS,
                                     null_psi_band(TEMPTING[:, :BASELINE_HOURS]),
                                     which="unplanned"))
    print_checklist(check_specification(my_specification(leaky_bundle), leaky_bundle),
                    "D · a different detector, documented just as carefully")
    ours = evidence()
    print(f"seven of its eight lines are true. Its backtest reports {point.cost:,.0f} "
          f"against your\n{ours.point.cost:,.0f} and it names "
          f"{len(leaky_bundle.given_up)} failures given up against your "
          f"{len(ours.given_up)} — and none of those numbers means\nanything, because the "
          f"feature they were computed on cannot be computed at the hour it\nclaims. That "
          f"is what line one is for. It is refused before anyone argues about the score.")


_try("the other detector, refused", _refuse_the_other_detector,
     needs=_EVIDENCE_NEEDS + ("exercise 10", "your specification"))

## 16. The handover note

One screen, seven lines, every number computed above. This is the document; the notebook is
the working.

In [ ]:
def _handover() -> None:
    bundle = evidence()
    spec = my_specification(bundle)
    results = check_specification(spec, bundle)
    print(f"MONITORING PROGRAMME — {spec.asset_class}, {N_UNITS} units\n")
    print(f"  feature       {spec.feature}")
    print(f"                causality margin {spec.causal_margin:.10f}, measured over a "
          f"{TAIL_HOURS} h probe")
    print(f"  labelling     {spec.labelling_policy}")
    print(f"                censors {bundle.censoring.n_censored} runs and "
          f"{bundle.censoring.unobserved_unit_hours:,} unit-hours of operation")
    print(f"  alarm rule    health index >= {spec.threshold:.2f}, and an alarm is a catch "
          f"only with {spec.lead_hours} h of warning")
    print(f"  prices        " + " · ".join(
        f"{n} {v:,.0f} ({p}, {c} jobs)" for n, v, p, c in
        zip(OUTCOMES, spec.prices, spec.price_provenance, bundle.prices.counts)))
    print(f"  expected      {bundle.point.counts.tp} caught, {bundle.point.counts.fn} "
          f"missed, {bundle.point.counts.fp} false alarms per {N_UNITS} pumps per "
          f"{N_HOURS} h, at {bundle.point.cost:,.0f}")
    print(f"  gives up      pumps " + ", ".join(
        f"{g.unit} ({g.reason}" + (f", {g.warning_hours} h of warning)" if g.warning_hours >= 0
                                   else ")") for g in bundle.given_up))
    print(f"  deployment    {spec.deployment_constraint}")
    print(f"                {spec.memory_bytes:,} bytes of arena, wire threshold "
          f"{bundle.deployment.wire_threshold:.6f}, "
          f"{bundle.deployment.units_whose_alarm_moves} pumps' first alarm moves")
    move = ("no move inside the scanned range" if np.isinf(spec.price_move_fraction)
            else f"the unplanned price moves by {100 * spec.price_move_fraction:.0f}%")
    print(f"  re-derive     when {move}, or the health index drifts past a PSI of "
          f"{spec.drift_alert:.5f}")
    print(f"\n  checklist     {sum(r.ok for r in results)} of {len(results)} items · "
          f"{'ACCEPTED' if specification_passes(results) else 'REJECTED'}")


_try("the handover note", _handover,
     needs=_EVIDENCE_NEEDS + ("exercise 10", "your specification"))

## 17. Common mistakes

- **Asserting a feature is causal.** It takes one function call to measure. A specification
  that says "causal" without a margin beside it has said nothing a reviewer can check.
- **Calling a suspension a healthy negative.** A pump pulled for a line reconfiguration did
  not survive; nobody watched it. Module 4's word is censored, and the number that makes it
  concrete is the unobserved unit-hours printed in section 6.
- **Booking the event at the hour the paperwork closed.** Section 6 measured the lag on this
  plant. It is a property of the maintenance office, not of any pump.
- **Averaging a price over two invoices.** A class below `MIN_OBSERVATIONS` keeps the
  kick-off guess and is reported as one. The alternative is a number with a decimal point
  and no evidence behind it.
- **Reporting a threshold without its prices.** Specification A in section 15 is otherwise
  perfect. A threshold is a function of three prices; without them it is a number.
- **Reporting a threshold without naming what it gives up.** Specification B. This is the
  one the whole programme is about.
- **Choosing the smoothing window offline.** Section 10 turned it into a memory budget. The
  cheapest window on this fleet does not fit the gateway, and the difference has a price.
- **Writing "re-derive when things change".** Section 11 turned that into two numbers: how
  far a price has to move, and what PSI counts as drift on this plant.
- **Treating the checklist as paperwork.** Run the next cell: it is the difference between a
  document that failed review and a document that failed review *before* you sent it.

In [ ]:
def _the_cost_of_skipping_the_checklist() -> None:
    bundle = evidence()
    mine = my_specification(bundle)
    skipped = mine._replace(prices=KICKOFF,
                            price_provenance=tuple(GUESSED for _ in OUTCOMES),
                            gives_up=())
    results = check_specification(skipped, bundle)
    failed = [r.item for r in results if not r.ok]
    kickoff_point = choose_operating_point(THRESHOLDS, SWEPT, KICKOFF)
    print(f"a specification with the kick-off prices and no give-ups fails "
          f"{len(failed)} of {len(results)} items:")
    for item in failed:
        print(f"    - {item}")
    print(f"\nand the threshold those kick-off prices actually support is "
          f"{kickoff_point.threshold:.2f}, not "
          f"{bundle.point.threshold:.2f}.")
    print(f"scored on this fleet at the audited prices, that threshold costs "
          f"{float(expected_cost(kickoff_point.counts, bundle.prices.prices)):,.0f} "
          f"against\n{bundle.point.cost:,.0f} — "
          f"{float(expected_cost(kickoff_point.counts, bundle.prices.prices)) / bundle.point.cost:.2f}x — "
          f"and gives up {kickoff_point.counts.fn} failures instead of "
          f"{bundle.point.counts.fn}.")
    print("the checklist did not find that. It found the two missing lines that would have "
          "let it\nthrough, which is the same thing three weeks earlier.")


_try("the cost of skipping the checklist", _the_cost_of_skipping_the_checklist,
     needs=_EVIDENCE_NEEDS + ("exercise 10", "your specification"))

## 18. Self-check

1. Your detector reaches an AUC of 0.97 on the backtest and a colleague's reaches 0.91.
   Yours smooths with a centred filter. The right conclusion is:
   - (a) ship yours; six points of AUC is a large margin
   - (b) ship theirs, because a lower score is the safer choice
   - (c) neither number means anything until the causality probe returns zero for both — a
         filter that reads hours ahead has not forecast anything

2. A specification reports the threshold, all three prices with their provenance, the lead
   time, the arena budget and the drift trigger, and states what share of the fleet's
   failures the programme catches. The checklist still refuses it. The missing line is:
   - (a) the ROC curve and its area
   - (b) which failures the threshold gives up, by unit, and why each one
   - (c) the detector's hyperparameters

3. The audit holds four false-alarm invoices against a minimum of five, so that price stays
   at the kick-off guess. The specification should:
   - (a) report that price as a guess on its own line, because the threshold is only as
         defensible as its weakest price
   - (b) average the four anyway; four invoices are better than none
   - (c) drop the false-alarm term from the cost function until more invoices arrive

Before you answer 4 and 5, run this. It prints the numbers those two questions turn on, so
your answer rests on the notebook's output rather than on a memory of it.

In [ ]:
def _self_check_numbers() -> None:
    bundle = evidence()
    ev = bundle.prices
    print(f"q4 · widest window the arena holds: "
          f"{(GATEWAY.arena_bytes // N_UNITS - GATEWAY.constant_bytes) // GATEWAY.sample_bytes} h "
          f"· the programme ships at {WINDOW} h using "
          f"{bundle.deployment.total_bytes:,} of {GATEWAY.arena_bytes:,} bytes")
    for name in OUTCOMES:
        trig = rederivation_trigger(THRESHOLDS, SWEPT, ev.prices, PRICE_MULTIPLIERS,
                                    bundle.trigger.drift_alert, which=name)
        span = (f"{100 * trig.price_move_fraction:.0f}%"
                if np.isfinite(trig.price_move_fraction)
                else f"inf (grid scanned +-{100 * (PRICE_MULTIPLIERS.max() - 1):.0f}%)")
        print(f"q5 · {name:<12} moves the threshold at {span}")


_try("self-check numbers", _self_check_numbers,
     needs=_EVIDENCE_NEEDS + ("exercise 10",))

4. The cheapest smoothing window on this fleet does not fit the gateway's arena. The right
   response is:
   - (a) ship the cheapest window and ask for a bigger gateway
   - (b) ignore the arena; the difference in cost is small and memory is cheap
   - (c) ship the widest window that fits, and put the difference **in currency** in the
         specification, so that the next person can price a bigger gateway against it

5. `rederivation_trigger` reports `inf` for the planned price. That means:
   - (a) the planned price can never move this threshold
   - (b) no multiplier in the range that was scanned moved it, which is a statement about
         the scan and not about the price
   - (c) the function failed and should have raised

Answers, with the reasoning, are in this lesson's worked solution in the course repository.

## What you built

A monitoring programme, and the document that makes it one. The runnable half is eight
functions and a fleet; the other half is eight lines that a person who has never seen this
notebook can check, and a checker that refuses the document when a line is missing.

The thing worth carrying out of this programme is the shape of the last refusal.
Specification D had a detector, a labelling policy, a lead time, three audited prices, a
named give-up list, an arena budget and a trigger. It was refused by one number that took
one function call to compute, and the number was not about the detector's quality at all —
it was about whether the detector could exist at the hour it claimed to run.

That is the difference between a model and a programme. A model is judged on how well it
separates two populations. A programme is judged on whether the number it ships can be
computed where it has to run, priced in the currency the plant actually spends, and
defended to the person who signs the work order — including the part where you tell them,
before anything breaks, which pumps are not on the list.

In [ ]:
import contextlib
import io

_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally.

    This grades your CODE. The PASS/FAIL lines of a checklist run grade a DOCUMENT; the two
    meet only at `your specification`, which is done when your document is accepted.
    """
    names = max(len(label) for label in _EXERCISES)
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<{names}}  {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check. Exercise 9 and
# `your specification` keep the `needs` their own cells have, so neither is credited, or
# blamed, on the strength of an exercise that has not passed yet.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_causal_health, ()),
                ("exercise 2", _check_causality_margin, ()),
                ("exercise 3", _check_label_runs, ()),
                ("exercise 4", _check_censoring_report, ()),
                ("exercise 5", _check_price_the_outcomes, ()),
                ("exercise 6", _check_choose_operating_point, ()),
                ("exercise 7", _check_failures_given_up, ()),
                ("exercise 8", _check_deployment_report, ()),
                ("exercise 9", _check_rederivation_trigger, ("exercise 6",)),
                ("exercise 10", _check_check_specification, ()),
                ("your specification", _run_my_checklist,
                 _EVIDENCE_NEEDS + ("exercise 10",))):
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))